# 08 — Confronti, ranking e visualizzazioni finali

Questo notebook organizza e visualizza i risultati prodotti dai notebook
precedenti.

Non vengono effettuate nuove inferenze e non vengono ricalcolate le metriche.

Gli obiettivi sono:

1. confrontare i cinque modelli sui sette task GLUE-style;
2. confrontare i modelli sulle tre tipologie tecniche di task:
   classificazione, numerico e generativo;
3. analizzare diversi livelli di zoom:
   - baseline contro tutti i fine-tuning;
   - confronto tra le tre varianti Legends;
   - confronto tra Regulation e le varianti Legends;
4. analizzare separatamente BLEU, ROUGE e BERTScore;
5. importare e visualizzare la valutazione soggettiva della generazione;
6. costruire il ranking automatico per task;
7. assegnare i punti da 5 a 1;
8. costruire il ranking classificatorio e quello complessivo;
9. mantenere gli approfondimenti diagnostici:
   - gains e regressions;
   - win, tie e loss rispetto alla baseline;
   - matrice di dominanza;
   - stabilità del ranking;
   - confusion matrix;
   - tipi di errore;
   - distribuzioni STS-B;
   - lunghezza delle risposte generative.

## Modelli analizzati

- `base`
- `regulation`
- `positive_legends`
- `negative_legends`
- `negative_to_correction`

## Task GLUE-style

- `sst2_style`
- `paraphrase_style`
- `qqp_style`
- `mnli_style`
- `rte_style`
- `bias_detection`
- `stsb_style`

## Task generativo

- `grounded_answer_generation`

## Principio del ranking

Le metriche eterogenee non vengono mediate direttamente.

Per ciascun task viene scelta una metrica rappresentativa. I cinque modelli
vengono ordinati e ricevono:

- 5 punti al primo;
- 4 punti al secondo;
- 3 punti al terzo;
- 2 punti al quarto;
- 1 punto al quinto.

In caso di parità viene assegnata la media dei punti delle posizioni occupate.

Il ranking è una vista sintetica finale e non sostituisce le analisi numeriche
e comparative precedenti.

In [ ]:
# CELLA 3 — Importa le librerie e definisce cartelle, modelli, task,
# ordine di visualizzazione e soglie numeriche utilizzate nel notebook.

from pathlib import Path
from itertools import combinations
from importlib.metadata import PackageNotFoundError, version

import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score


# Modificare solo questo percorso se il progetto si trova altrove.
PROJECT_ROOT = Path(
    r"C:\Users\fede2\Desktop\Project"
).resolve()


METRICS_DIR = PROJECT_ROOT / "data" / "metrics"
ANALYSIS_DIR = PROJECT_ROOT / "data" / "analysis"

FINAL_TABLES_DIR = (
    PROJECT_ROOT
    / "data"
    / "final_tables"
)

TASK_TABLES_DIR = (
    FINAL_TABLES_DIR
    / "task_tables"
)

VISUALIZATIONS_DIR = (
    PROJECT_ROOT
    / "data"
    / "visualizations"
)

TASK_PLOTS_DIR = (
    VISUALIZATIONS_DIR
    / "task_comparisons"
)

ZOOM_PLOTS_DIR = (
    VISUALIZATIONS_DIR
    / "comparison_zooms"
)

GENERATION_PLOTS_DIR = (
    VISUALIZATIONS_DIR
    / "generation"
)

SUBJECTIVE_PLOTS_DIR = (
    VISUALIZATIONS_DIR
    / "subjective"
)

RANKING_PLOTS_DIR = (
    VISUALIZATIONS_DIR
    / "rankings"
)

CONFUSION_MATRIX_DIR = (
    VISUALIZATIONS_DIR
    / "confusion_matrices"
)

ERROR_TYPE_DIR = (
    VISUALIZATIONS_DIR
    / "error_types"
)

STSB_PLOTS_DIR = (
    VISUALIZATIONS_DIR
    / "stsb"
)


for directory in [
    FINAL_TABLES_DIR,
    TASK_TABLES_DIR,
    VISUALIZATIONS_DIR,
    TASK_PLOTS_DIR,
    ZOOM_PLOTS_DIR,
    GENERATION_PLOTS_DIR,
    SUBJECTIVE_PLOTS_DIR,
    RANKING_PLOTS_DIR,
    CONFUSION_MATRIX_DIR,
    ERROR_TYPE_DIR,
    STSB_PLOTS_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


MODEL_ORDER = [
    "base",
    "regulation",
    "positive_legends",
    "negative_legends",
    "negative_to_correction",
]

BASE_MODEL = "base"

ADAPTER_MODELS = [
    model_name
    for model_name in MODEL_ORDER
    if model_name != BASE_MODEL
]

LEGEND_MODELS = [
    "positive_legends",
    "negative_legends",
    "negative_to_correction",
]

REGULATION_AND_LEGENDS_MODELS = [
    "regulation",
    "positive_legends",
    "negative_legends",
    "negative_to_correction",
]


MODEL_ORDER_MAP = {
    model_name: index
    for index, model_name in enumerate(
        MODEL_ORDER
    )
}


CLASSIFICATION_TASKS = [
    "sst2_style",
    "paraphrase_style",
    "qqp_style",
    "mnli_style",
    "rte_style",
    "bias_detection",
]

STSB_TASK = "stsb_style"

GENERATION_TASK = (
    "grounded_answer_generation"
)

GLUE_TASKS = (
    CLASSIFICATION_TASKS
    + [STSB_TASK]
)

ALL_TASKS = (
    GLUE_TASKS
    + [GENERATION_TASK]
)


TASK_DISPLAY_NAMES = {
    "sst2_style": "SST-2",
    "paraphrase_style": "Paraphrase",
    "qqp_style": "QQP",
    "mnli_style": "MNLI",
    "rte_style": "RTE",
    "bias_detection": "Bias",
    "stsb_style": "STS-B",
    "grounded_answer_generation": "Generation",
}


TASK_ORDER_MAP = {
    task_name: index
    for index, task_name in enumerate(
        ALL_TASKS
    )
}


TASK_LABELS = {
    "sst2_style": [
        "compliant",
        "non_compliant",
    ],

    "paraphrase_style": [
        "equivalent",
        "not_equivalent",
    ],

    "qqp_style": [
        "duplicate",
        "not_duplicate",
    ],

    "mnli_style": [
        "entailment",
        "contradiction",
        "neutral",
    ],

    "rte_style": [
        "entailment",
        "not_entailment",
    ],

    "bias_detection": [
        "biased",
        "unbiased",
    ],
}


RANKING_TOLERANCE = 1e-12

EXPECTED_EXAMPLES_PER_TASK = 20

EXPECTED_GENERATION_ROWS = (
    len(MODEL_ORDER)
    * EXPECTED_EXAMPLES_PER_TASK
)


print("Metriche:", METRICS_DIR)
print("Analisi:", ANALYSIS_DIR)
print("Tabelle finali:", FINAL_TABLES_DIR)
print("Grafici:", VISUALIZATIONS_DIR)

In [ ]:
# CELLA 4 — Definisce i file prodotti dai notebook 05, 06 e 07
# utilizzati per costruire confronti, ranking e visualizzazioni.

INPUT_PATHS = {
    "primary_task_scores": (
        METRICS_DIR
        / "primary_task_scores.csv"
    ),

    "classification_results": (
        METRICS_DIR
        / "classification_results_by_example.csv"
    ),

    "stsb_results": (
        METRICS_DIR
        / "stsb_results_by_example.csv"
    ),

    "generation_by_example": (
        METRICS_DIR
        / "generation_metrics_by_example.csv"
    ),

    "generation_by_model": (
        METRICS_DIR
        / "generation_metrics_by_model.csv"
    ),

    "fine_grained_metrics": (
        ANALYSIS_DIR
        / "fine_grained_metrics_all_tasks.csv"
    ),

    "classification_adapter_vs_base_summary": (
        ANALYSIS_DIR
        / "classification_adapter_vs_base_summary.csv"
    ),

    "classification_error_adapter_vs_base": (
        ANALYSIS_DIR
        / "classification_error_adapter_vs_base.csv"
    ),

    "classification_intermediate_by_model": (
        ANALYSIS_DIR
        / "classification_intermediate_by_model.csv"
    ),

    "stsb_distance_profile": (
        ANALYSIS_DIR
        / "stsb_distance_profile_by_model.csv"
    ),
}


SUBJECTIVE_FILE_CANDIDATES = [
    (
        METRICS_DIR
        / "subjective_evaluation_completed.xlsx"
    ),

    (
        METRICS_DIR
        / "subjective_evaluation_to_complete.xlsx"
    ),
]


missing_input_files = [
    str(path)
    for path in INPUT_PATHS.values()
    if not path.exists()
]


if missing_input_files:
    raise FileNotFoundError(
        "Mancano alcuni file obbligatori:\n- "
        + "\n- ".join(
            missing_input_files
        )
    )


print("Tutti i file obbligatori sono disponibili.")

for input_name, input_path in INPUT_PATHS.items():
    print(f"- {input_name}: {input_path}")

In [ ]:
# CELLA 5 — Carica i CSV prodotti dai notebook precedenti.
# Nessuna metrica viene ricalcolata in questa cella.

primary_task_scores_df = pd.read_csv(
    INPUT_PATHS[
        "primary_task_scores"
    ]
)

classification_results_df = pd.read_csv(
    INPUT_PATHS[
        "classification_results"
    ]
)

stsb_results_df = pd.read_csv(
    INPUT_PATHS[
        "stsb_results"
    ]
)

generation_by_example_df = pd.read_csv(
    INPUT_PATHS[
        "generation_by_example"
    ]
)

generation_by_model_df = pd.read_csv(
    INPUT_PATHS[
        "generation_by_model"
    ]
)

fine_grained_metrics_df = pd.read_csv(
    INPUT_PATHS[
        "fine_grained_metrics"
    ]
)

classification_adapter_vs_base_summary_df = (
    pd.read_csv(
        INPUT_PATHS[
            "classification_adapter_vs_base_summary"
        ]
    )
)

classification_error_adapter_vs_base_df = (
    pd.read_csv(
        INPUT_PATHS[
            "classification_error_adapter_vs_base"
        ]
    )
)

classification_intermediate_by_model_df = (
    pd.read_csv(
        INPUT_PATHS[
            "classification_intermediate_by_model"
        ]
    )
)

stsb_distance_profile_df = pd.read_csv(
    INPUT_PATHS[
        "stsb_distance_profile"
    ]
)


print(
    "Primary score non generativi:",
    len(primary_task_scores_df),
)

print(
    "Predizioni classificatorie:",
    len(classification_results_df),
)

print(
    "Predizioni STS-B:",
    len(stsb_results_df),
)

print(
    "Risposte generative:",
    len(generation_by_example_df),
)

In [ ]:
# CELLA 6 — Controlla che i cinque modelli siano presenti nei principali
# dataframe e che il numero delle risposte generative sia quello atteso.

def validate_models(
    dataframe: pd.DataFrame,
    dataframe_name: str,
):
    """Verifica che siano presenti esattamente i cinque modelli attesi."""

    if "model_name" not in dataframe.columns:
        raise RuntimeError(
            f"{dataframe_name}: colonna model_name mancante."
        )

    actual_models = set(
        dataframe[
            "model_name"
        ]
        .astype(str)
        .str.strip()
    )

    missing_models = (
        set(MODEL_ORDER)
        - actual_models
    )

    unexpected_models = (
        actual_models
        - set(MODEL_ORDER)
    )

    if missing_models:
        raise RuntimeError(
            f"{dataframe_name}: modelli mancanti: "
            f"{sorted(missing_models)}"
        )

    if unexpected_models:
        raise RuntimeError(
            f"{dataframe_name}: modelli inattesi: "
            f"{sorted(unexpected_models)}"
        )


for dataframe, dataframe_name in [
    (
        primary_task_scores_df,
        "primary_task_scores",
    ),
    (
        classification_results_df,
        "classification_results",
    ),
    (
        stsb_results_df,
        "stsb_results",
    ),
    (
        generation_by_example_df,
        "generation_by_example",
    ),
    (
        generation_by_model_df,
        "generation_by_model",
    ),
    (
        fine_grained_metrics_df,
        "fine_grained_metrics",
    ),
]:
    validate_models(
        dataframe,
        dataframe_name,
    )


if (
    len(generation_by_example_df)
    != EXPECTED_GENERATION_ROWS
):
    raise RuntimeError(
        "Numero inatteso di risposte generative: "
        f"{len(generation_by_example_df)} invece di "
        f"{EXPECTED_GENERATION_ROWS}."
    )


print("Validazione degli input completata.")

In [ ]:
# CELLA 7 — Definisce le funzioni comuni per salvare i grafici,
# aggiungere griglie, annotare heatmap e ordinare i modelli.

def save_figure(
    figure: plt.Figure,
    output_path: Path,
):
    """Salva la figura e la chiude per liberare memoria."""

    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=180,
        bbox_inches="tight",
    )

    plt.show()


def apply_y_grid(
    axis: plt.Axes,
):
    """Aggiunge una griglia orizzontale leggera."""

    axis.grid(
        axis="y",
        alpha=0.25,
    )

    axis.set_axisbelow(
        True
    )


def apply_x_grid(
    axis: plt.Axes,
):
    """Aggiunge una griglia verticale leggera."""

    axis.grid(
        axis="x",
        alpha=0.25,
    )

    axis.set_axisbelow(
        True
    )


def annotate_heatmap(
    axis: plt.Axes,
    matrix: np.ndarray,
    decimals: int = 3,
):
    """Scrive il valore numerico all'interno delle celle."""

    for row_index in range(
        matrix.shape[0]
    ):
        for column_index in range(
            matrix.shape[1]
        ):
            value = matrix[
                row_index,
                column_index,
            ]

            if np.isnan(value):
                text_value = ""

            else:
                text_value = (
                    f"{value:.{decimals}f}"
                )

            axis.text(
                column_index,
                row_index,
                text_value,
                ha="center",
                va="center",
            )


def sort_models(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """Ordina un dataframe secondo MODEL_ORDER."""

    sorted_dataframe = dataframe.copy()

    sorted_dataframe[
        "_model_order"
    ] = (
        sorted_dataframe[
            "model_name"
        ]
        .map(MODEL_ORDER_MAP)
    )

    return (
        sorted_dataframe
        .sort_values(
            "_model_order"
        )
        .drop(
            columns="_model_order"
        )
        .reset_index(
            drop=True
        )
    )
#AGGIUNGERE CELLA PER LE ALTRE METRICHE

# Sezione 1 — Preparazione delle metriche rappresentative

Per ogni task viene utilizzata una metrica rappresentativa:

| Task | Metrica |
|---|---|
| SST-2 | Accuracy |
| Paraphrase | Media tra accuracy e F1 equivalent |
| QQP | Media tra accuracy e F1 duplicate |
| MNLI | Accuracy |
| RTE | Accuracy |
| Bias | Macro-F1 |
| STS-B | Media tra Pearson e Spearman |
| Generazione | BERTScore F1 |

Questi valori vengono usati per:

- la tabella GLUE-style 5×7;
- i profili a linee;
- i confronti tra modelli;
- il ranking automatico.

Le metriche dettagliate dei singoli task rimangono comunque disponibili nelle
tabelle fine-grained.

In [ ]:
# CELLA 9 — Unisce i primary score dei sette task GLUE-style
# con BERTScore F1 del task generativo.

automatic_primary_scores_df = (
    primary_task_scores_df[
        [
            "model_name",
            "task_name",
            "primary_metric_name",
            "primary_score",
        ]
    ]
    .copy()
)


generation_primary_scores_df = (
    generation_by_model_df[
        [
            "model_name",
            "mean_bertscore_f1",
        ]
    ]
    .rename(
        columns={
            "mean_bertscore_f1": (
                "primary_score"
            )
        }
    )
)


generation_primary_scores_df[
    "task_name"
] = GENERATION_TASK

generation_primary_scores_df[
    "primary_metric_name"
] = "mean_bertscore_f1"


automatic_primary_scores_df = pd.concat(
    [
        automatic_primary_scores_df,

        generation_primary_scores_df[
            [
                "model_name",
                "task_name",
                "primary_metric_name",
                "primary_score",
            ]
        ],
    ],
    ignore_index=True,
)


automatic_primary_scores_df[
    "_task_order"
] = (
    automatic_primary_scores_df[
        "task_name"
    ]
    .map(TASK_ORDER_MAP)
)

automatic_primary_scores_df[
    "_model_order"
] = (
    automatic_primary_scores_df[
        "model_name"
    ]
    .map(MODEL_ORDER_MAP)
)

automatic_primary_scores_df = (
    automatic_primary_scores_df
    .sort_values(
        [
            "_task_order",
            "_model_order",
        ]
    )
    .drop(
        columns=[
            "_task_order",
            "_model_order",
        ]
    )
    .reset_index(
        drop=True
    )
)


expected_primary_rows = (
    len(MODEL_ORDER)
    * len(ALL_TASKS)
)

if (
    len(automatic_primary_scores_df)
    != expected_primary_rows
):
    raise RuntimeError(
        "Numero inatteso di primary score: "
        f"{len(automatic_primary_scores_df)} invece di "
        f"{expected_primary_rows}."
    )


display(
    automatic_primary_scores_df
    .round(4)
)

In [ ]:
# CELLA 10 — Definisce la funzione che ordina i modelli per task,
# gestisce le parità e assegna i punti da 5 a 1.

def rank_models_for_task(
    task_dataframe: pd.DataFrame,
    score_column: str = "primary_score",
    tolerance: float = RANKING_TOLERANCE,
) -> pd.DataFrame:
    """Ordina i modelli e assegna punti preservando le parità."""

    ranked = (
        task_dataframe
        .sort_values(
            score_column,
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )

    result_rows = []

    current_index = 0
    current_position = 1

    number_of_models = len(
        ranked
    )


    while current_index < number_of_models:
        reference_score = float(
            ranked.loc[
                current_index,
                score_column,
            ]
        )

        tie_indices = [
            current_index
        ]

        next_index = (
            current_index
            + 1
        )


        while next_index < number_of_models:
            next_score = float(
                ranked.loc[
                    next_index,
                    score_column,
                ]
            )

            if np.isclose(
                reference_score,
                next_score,
                atol=tolerance,
                rtol=0,
            ):
                tie_indices.append(
                    next_index
                )

                next_index += 1

            else:
                break


        tie_size = len(
            tie_indices
        )

        occupied_positions = list(
            range(
                current_position,
                current_position
                + tie_size,
            )
        )

        occupied_points = [
            number_of_models
            + 1
            - position
            for position in occupied_positions
        ]

        average_points = float(
            np.mean(
                occupied_points
            )
        )

        average_rank = float(
            np.mean(
                occupied_positions
            )
        )

        rank_end = int(
            occupied_positions[-1]
        )


        for row_index in tie_indices:
            row_data = (
                ranked.loc[
                    row_index
                ]
                .to_dict()
            )

            row_data.update({
                "rank": int(
                    current_position
                ),

                "rank_end": (
                    rank_end
                ),

                "average_rank": (
                    average_rank
                ),

                "points": (
                    average_points
                ),

                "tie_size": (
                    tie_size
                ),

                "ranking_status": (
                    "tie"
                    if tie_size > 1
                    else "unique_position"
                ),
            })

            result_rows.append(
                row_data
            )


        current_index = next_index
        current_position += tie_size


    return pd.DataFrame(
        result_rows
    )

In [ ]:
# CELLA 11 — Applica il ranking ai sette task GLUE-style
# e al task generativo.

ranking_parts = []


for task_name in ALL_TASKS:
    task_group = (
        automatic_primary_scores_df[
            automatic_primary_scores_df[
                "task_name"
            ]
            == task_name
        ]
        .copy()
    )

    ranking_parts.append(
        rank_models_for_task(
            task_group
        )
    )


task_rankings_automatic_df = pd.concat(
    ranking_parts,
    ignore_index=True,
)


task_rankings_automatic_df[
    "_task_order"
] = (
    task_rankings_automatic_df[
        "task_name"
    ]
    .map(TASK_ORDER_MAP)
)

task_rankings_automatic_df = (
    task_rankings_automatic_df
    .sort_values(
        [
            "_task_order",
            "rank",
            "model_name",
        ]
    )
    .drop(
        columns="_task_order"
    )
    .reset_index(
        drop=True
    )
)


display(
    task_rankings_automatic_df[
        [
            "task_name",
            "model_name",
            "primary_metric_name",
            "primary_score",
            "rank",
            "average_rank",
            "points",
            "ranking_status",
        ]
    ]
    .round(4)
)

In [ ]:
# CELLA 13 — Costruisce la matrice 5×7 dei primary score:
# cinque modelli per sette task GLUE-style.

glue_primary_scores_matrix_df = (
    automatic_primary_scores_df[
        automatic_primary_scores_df[
            "task_name"
        ]
        .isin(
            GLUE_TASKS
        )
    ]
    .pivot(
        index="model_name",
        columns="task_name",
        values="primary_score",
    )
    .reindex(
        index=MODEL_ORDER,
        columns=GLUE_TASKS,
    )
)


glue_primary_scores_matrix_df.columns = [
    TASK_DISPLAY_NAMES[
        task_name
    ]
    for task_name in GLUE_TASKS
]


display(
    glue_primary_scores_matrix_df
    .round(4)
)

In [ ]:
# CELLA 14 — Genera la heatmap dei primary score
# dei cinque modelli sui sette task GLUE-style.

glue_matrix = (
    glue_primary_scores_matrix_df
    .to_numpy(
        dtype=float
    )
)


figure, axis = plt.subplots(
    figsize=(12, 5)
)

image = axis.imshow(
    glue_matrix,
    aspect="auto",
)


axis.set_title(
    "Confronto GLUE-style: primary score per modello e task"
)

axis.set_xlabel(
    "Task"
)

axis.set_ylabel(
    "Modello"
)


axis.set_xticks(
    np.arange(
        len(GLUE_TASKS)
    )
)

axis.set_xticklabels(
    glue_primary_scores_matrix_df.columns,
    rotation=30,
    ha="right",
)


axis.set_yticks(
    np.arange(
        len(MODEL_ORDER)
    )
)

axis.set_yticklabels(
    MODEL_ORDER
)


annotate_heatmap(
    axis,
    glue_matrix,
    decimals=3,
)


figure.colorbar(
    image,
    ax=axis,
    label="Primary score",
)


save_figure(
    figure,
    TASK_PLOTS_DIR
    / "glue_primary_scores_heatmap_5x7.png",
)

In [ ]:
# CELLA 15 — Genera un profilo a linee con una linea per modello
# lungo i sette task GLUE-style.

figure, axis = plt.subplots(
    figsize=(13, 6)
)


x_positions = np.arange(
    len(GLUE_TASKS)
)


for model_name in MODEL_ORDER:
    model_values = (
        glue_primary_scores_matrix_df
        .loc[
            model_name
        ]
        .to_numpy(
            dtype=float
        )
    )

    axis.plot(
        x_positions,
        model_values,
        marker="o",
        linewidth=2,
        label=model_name,
    )


axis.set_title(
    "Profilo delle prestazioni sui task GLUE-style"
)

axis.set_xlabel(
    "Task"
)

axis.set_ylabel(
    "Primary score"
)


axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    glue_primary_scores_matrix_df.columns,
    rotation=30,
    ha="right",
)


axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    TASK_PLOTS_DIR
    / "glue_model_profiles_lines.png",
)

In [ ]:
# CELLA 16 — Genera un grafico a barre separato per ciascun task,
# confrontando direttamente i cinque modelli.

for task_name in GLUE_TASKS:
    task_data = (
        automatic_primary_scores_df[
            automatic_primary_scores_df[
                "task_name"
            ]
            == task_name
        ]
        .copy()
    )

    task_data = sort_models(
        task_data
    )


    figure, axis = plt.subplots(
        figsize=(9, 5)
    )


    axis.bar(
        task_data[
            "model_name"
        ],
        task_data[
            "primary_score"
        ],
    )


    axis.set_title(
        f"{TASK_DISPLAY_NAMES[task_name]} — confronto dei modelli"
    )

    axis.set_xlabel(
        "Modello"
    )

    axis.set_ylabel(
        task_data[
            "primary_metric_name"
        ]
        .iloc[0]
    )

    axis.tick_params(
        axis="x",
        rotation=25,
    )


    apply_y_grid(
        axis
    )


    save_figure(
        figure,
        TASK_PLOTS_DIR
        / f"{task_name}_model_comparison_bar.png",
    )

    # RIDURRE SCALA E CAMBIARE COLORI

In [ ]:
# CELLA 17 — Esporta e visualizza le metriche dettagliate
# dei cinque modelli per ogni task GLUE-style.

for task_name in GLUE_TASKS:
    task_table = (
        fine_grained_metrics_df[
            fine_grained_metrics_df[
                "task_name"
            ]
            == task_name
        ]
        .copy()
    )

    task_table = sort_models(
        task_table
    )


    task_table.to_csv(
        TASK_TABLES_DIR
        / f"{task_name}_detailed_metrics.csv",
        index=False,
        encoding="utf-8-sig",
    )


    print("\n" + "=" * 90)
    print(
        f"Metriche dettagliate: "
        f"{TASK_DISPLAY_NAMES[task_name]}"
    )
    print("=" * 90)

    display(
        task_table
        .round(4)
    )
    #fintunare le colonne per ogni subtask non va bene così tanti nan

# Sezione 3 — Confronto per tipologia tecnica 5×3

I task vengono raggruppati in tre tipologie:

1. classificazione;
2. numerico, rappresentato da STS-B;
3. generativo.

Per evitare di mediare metriche eterogenee, il confronto viene effettuato
sulla scala comune dei punti da 1 a 5.

- Classificazione: media dei punti sui sei task classificatori.
- Numerico: punti ottenuti su STS-B.
- Generativo: punti ottenuti sulla generazione automatica.

Vengono prodotti:

- tabella 5×3;
- grafico a barre raggruppate;
- profilo a linee.

In [ ]:
# CELLA 19 — Costruisce la tabella 5×3 utilizzando i punti medi
# ottenuti nelle tre tipologie tecniche di task.

classification_points_df = (
    task_rankings_automatic_df[
        task_rankings_automatic_df[
            "task_name"
        ]
        .isin(
            CLASSIFICATION_TASKS
        )
    ]
    .groupby(
        "model_name",
        as_index=False,
    )
    .agg(
        classification=(
            "points",
            "mean",
        )
    )
)


numeric_points_df = (
    task_rankings_automatic_df[
        task_rankings_automatic_df[
            "task_name"
        ]
        == STSB_TASK
    ]
    [
        [
            "model_name",
            "points",
        ]
    ]
    .rename(
        columns={
            "points": "numeric"
        }
    )
)


generative_points_df = (
    task_rankings_automatic_df[
        task_rankings_automatic_df[
            "task_name"
        ]
        == GENERATION_TASK
    ]
    [
        [
            "model_name",
            "points",
        ]
    ]
    .rename(
        columns={
            "points": "generative"
        }
    )
)


task_type_summary_df = (
    classification_points_df
    .merge(
        numeric_points_df,
        on="model_name",
        validate="one_to_one",
    )
    .merge(
        generative_points_df,
        on="model_name",
        validate="one_to_one",
    )
)


task_type_summary_df = sort_models(
    task_type_summary_df
)


display(
    task_type_summary_df
    .round(3)
)

#COSA PORCODIO SONO I PUNTI MEDI 5X3

In [ ]:
# CELLA 20 — Visualizza i punti medi delle tre tipologie
# con tre barre per ciascun modello.

figure, axis = plt.subplots(
    figsize=(12, 6)
)


x_positions = np.arange(
    len(MODEL_ORDER)
)

bar_width = 0.25


axis.bar(
    x_positions
    - bar_width,
    task_type_summary_df[
        "classification"
    ],
    width=bar_width,
    label="Classificazione",
)

axis.bar(
    x_positions,
    task_type_summary_df[
        "numeric"
    ],
    width=bar_width,
    label="Numerico / STS-B",
)

axis.bar(
    x_positions
    + bar_width,
    task_type_summary_df[
        "generative"
    ],
    width=bar_width,
    label="Generativo",
)


axis.set_title(
    "Confronto per tipologia tecnica di task"
)

axis.set_xlabel(
    "Modello"
)

axis.set_ylabel(
    "Punti medi su scala 1–5"
)


axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    MODEL_ORDER,
    rotation=25,
    ha="right",
)


axis.set_ylim(
    0,
    5.5,
)

axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    TASK_PLOTS_DIR
    / "task_type_comparison_grouped_bars_5x3.png",
)

In [ ]:
# CELLA 21 — Visualizza il profilo di ciascun modello
# sulle tre tipologie tecniche.

task_type_columns = [
    "classification",
    "numeric",
    "generative",
]

task_type_labels = [
    "Classificazione",
    "Numerico",
    "Generativo",
]


figure, axis = plt.subplots(
    figsize=(10, 6)
)


x_positions = np.arange(
    len(task_type_columns)
)


for row in task_type_summary_df.itertuples(
    index=False
):
    model_values = [
        row.classification,
        row.numeric,
        row.generative,
    ]

    axis.plot(
        x_positions,
        model_values,
        marker="o",
        linewidth=2,
        label=row.model_name,
    )


axis.set_title(
    "Profilo dei modelli per tipologia di task"
)

axis.set_xlabel(
    "Tipologia"
)

axis.set_ylabel(
    "Punti medi su scala 1–5"
)


axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    task_type_labels
)

axis.set_ylim(
    0,
    5.5,
)

axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    TASK_PLOTS_DIR
    / "task_type_comparison_lines_5x3.png",
)

# Sezione 4 — Zoom 1: baseline contro modelli fine-tunati

Questa sezione confronta il modello `base` con:

- `regulation`;
- `positive_legends`;
- `negative_legends`;
- `negative_to_correction`.

Vengono prodotti:

1. tabella dei valori assoluti;
2. tabella dei delta rispetto alla baseline;
3. grafico a linee dei valori assoluti;
4. grafico a barre dei delta;
5. gains, regressions e net gain;
6. task migliorati, invariati e peggiorati.

In [ ]:
# CELLA 23 — Costruisce le tabelle dei primary score assoluti
# e dei delta di ciascun adapter rispetto alla baseline.

all_primary_matrix_df = (
    automatic_primary_scores_df
    .pivot(
        index="task_name",
        columns="model_name",
        values="primary_score",
    )
    .reindex(
        ALL_TASKS
    )
)


base_vs_finetuning_absolute_df = (
    all_primary_matrix_df[
        MODEL_ORDER
    ]
    .copy()
)


base_vs_finetuning_delta_df = pd.DataFrame(
    index=ALL_TASKS
)


for adapter_name in ADAPTER_MODELS:
    base_vs_finetuning_delta_df[
        adapter_name
    ] = (
        all_primary_matrix_df[
            adapter_name
        ]
        - all_primary_matrix_df[
            BASE_MODEL
        ]
    )


display(
    base_vs_finetuning_absolute_df
    .round(4)
)

display(
    base_vs_finetuning_delta_df
    .round(4)
)

In [ ]:
# CELLA 24 — Genera il grafico a linee dei primary score
# del modello base e dei quattro fine-tuning.

figure, axis = plt.subplots(
    figsize=(14, 6)
)


x_positions = np.arange(
    len(ALL_TASKS)
)


for model_name in MODEL_ORDER:
    axis.plot(
        x_positions,
        base_vs_finetuning_absolute_df[
            model_name
        ],
        marker="o",
        linewidth=2,
        label=model_name,
    )


axis.set_title(
    "Baseline e modelli fine-tunati: profilo delle prestazioni"
)

axis.set_xlabel(
    "Task"
)

axis.set_ylabel(
    "Primary score"
)


axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    [
        TASK_DISPLAY_NAMES[
            task_name
        ]
        for task_name in ALL_TASKS
    ],
    rotation=30,
    ha="right",
)


axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    ZOOM_PLOTS_DIR
    / "base_vs_finetuning_absolute_lines.png",
)

In [ ]:
# CELLA 25 — Genera il grafico a barre dei delta:
# adapter score meno base score per ciascun task.

figure, axis = plt.subplots(
    figsize=(14, 6)
)


x_positions = np.arange(
    len(ALL_TASKS)
)

bar_width = (
    0.8
    / len(ADAPTER_MODELS)
)


for adapter_index, adapter_name in enumerate(
    ADAPTER_MODELS
):
    axis.bar(
        x_positions
        + adapter_index
        * bar_width
        - 0.4
        + bar_width / 2,
        base_vs_finetuning_delta_df[
            adapter_name
        ],
        width=bar_width,
        label=adapter_name,
    )


axis.axhline(
    0,
    linewidth=1,
)


axis.set_title(
    "Delta dei modelli fine-tunati rispetto alla baseline"
)

axis.set_xlabel(
    "Task"
)

axis.set_ylabel(
    "Primary score adapter − base"
)


axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    [
        TASK_DISPLAY_NAMES[
            task_name
        ]
        for task_name in ALL_TASKS
    ],
    rotation=30,
    ha="right",
)


axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    ZOOM_PLOTS_DIR
    / "base_vs_finetuning_delta_bars.png",
)

In [ ]:
# CELLA 26 — Visualizza gains, regressions e net gain
# dei quattro adapter sui 120 esempi classificatori.

classification_behavior_df = (
    classification_intermediate_by_model_df[
        classification_intermediate_by_model_df[
            "model_name"
        ]
        .isin(
            ADAPTER_MODELS
        )
    ]
    .copy()
)

classification_behavior_df = sort_models(
    classification_behavior_df
)


figure, axis = plt.subplots(
    figsize=(11, 6)
)


x_positions = np.arange(
    len(ADAPTER_MODELS)
)

bar_width = 0.25


axis.bar(
    x_positions
    - bar_width,
    classification_behavior_df[
        "gain"
    ],
    width=bar_width,
    label="Gains",
)

axis.bar(
    x_positions,
    classification_behavior_df[
        "regression"
    ],
    width=bar_width,
    label="Regressions",
)

axis.bar(
    x_positions
    + bar_width,
    classification_behavior_df[
        "classification_net_gain"
    ],
    width=bar_width,
    label="Net gain",
)


axis.axhline(
    0,
    linewidth=1,
)


axis.set_title(
    "Gains e regressions rispetto alla baseline"
)

axis.set_xlabel(
    "Adapter"
)

axis.set_ylabel(
    "Numero di esempi"
)


axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    ADAPTER_MODELS,
    rotation=25,
    ha="right",
)


axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    ZOOM_PLOTS_DIR
    / "base_vs_finetuning_gains_regressions.png",
)

In [ ]:
# CELLA 27 — Genera un grafico per ciascun adapter
# con gains e regressions distinti per task classificatorio.

for adapter_name in ADAPTER_MODELS:
    adapter_data = (
        classification_adapter_vs_base_summary_df[
            classification_adapter_vs_base_summary_df[
                "adapter_name"
            ]
            == adapter_name
        ]
        .set_index("task_name")
        .reindex(
            CLASSIFICATION_TASKS
        )
        .reset_index()
    )


    figure, axis = plt.subplots(
        figsize=(11, 5)
    )


    x_positions = np.arange(
        len(CLASSIFICATION_TASKS)
    )

    bar_width = 0.35


    axis.bar(
        x_positions
        - bar_width / 2,
        adapter_data[
            "gain"
        ],
        width=bar_width,
        label="Gains",
    )

    axis.bar(
        x_positions
        + bar_width / 2,
        adapter_data[
            "regression"
        ],
        width=bar_width,
        label="Regressions",
    )


    axis.set_title(
        f"Gains e regressions per task — {adapter_name}"
    )

    axis.set_xlabel(
        "Task"
    )

    axis.set_ylabel(
        "Numero di esempi"
    )


    axis.set_xticks(
        x_positions
    )

    axis.set_xticklabels(
        [
            TASK_DISPLAY_NAMES[
                task_name
            ]
            for task_name in CLASSIFICATION_TASKS
        ],
        rotation=30,
        ha="right",
    )


    axis.legend()

    apply_y_grid(
        axis
    )


    save_figure(
        figure,
        ZOOM_PLOTS_DIR
        / f"gains_regressions_{adapter_name}.png",
    )

In [ ]:
# CELLA 28 — Conta per ciascun adapter quanti task risultano
# migliorati, invariati o peggiorati rispetto al modello base.

win_tie_loss_rows = []


for adapter_name in ADAPTER_MODELS:
    adapter_deltas = (
        base_vs_finetuning_delta_df[
            adapter_name
        ]
    )

    win_tie_loss_rows.append({
        "model_name": adapter_name,

        "wins": int(
            (
                adapter_deltas
                > RANKING_TOLERANCE
            )
            .sum()
        ),

        "ties": int(
            np.isclose(
                adapter_deltas,
                0,
                atol=RANKING_TOLERANCE,
                rtol=0,
            )
            .sum()
        ),

        "losses": int(
            (
                adapter_deltas
                < -RANKING_TOLERANCE
            )
            .sum()
        ),
    })


win_tie_loss_vs_base_df = pd.DataFrame(
    win_tie_loss_rows
)


figure, axis = plt.subplots(
    figsize=(10, 6)
)


bottom_values = np.zeros(
    len(ADAPTER_MODELS)
)


for column_name in [
    "wins",
    "ties",
    "losses",
]:
    values = (
        win_tie_loss_vs_base_df[
            column_name
        ]
        .to_numpy(
            dtype=float
        )
    )

    axis.bar(
        win_tie_loss_vs_base_df[
            "model_name"
        ],
        values,
        bottom=bottom_values,
        label=column_name,
    )

    bottom_values += values


axis.set_title(
    "Task migliorati, invariati e peggiorati rispetto alla baseline"
)

axis.set_xlabel(
    "Adapter"
)

axis.set_ylabel(
    "Numero di task"
)

axis.tick_params(
    axis="x",
    rotation=25,
)

axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    ZOOM_PLOTS_DIR
    / "win_tie_loss_vs_base.png",
)

# Sezione 5 — Zoom 2: confronto tra le varianti Legends

Questa sezione confronta:

- `positive_legends`;
- `negative_legends`;
- `negative_to_correction`.

L'obiettivo è capire se la modalità con cui la conoscenza normativa viene
rappresentata influenza diversamente i task.

Vengono prodotti:

- tabella numerica;
- profilo a linee;
- grafico a barre raggruppate;
- sintesi dei punti e dei piazzamenti;
- numero di task vinti nel confronto interno.

In [ ]:
# CELLA 30 — Costruisce la tabella dei primary score
# delle tre varianti Legends su tutti gli otto task.

legend_variants_comparison_df = (
    all_primary_matrix_df[
        LEGEND_MODELS
    ]
    .copy()
)


display(
    legend_variants_comparison_df
    .round(4)
)

In [ ]:
# CELLA 31 — Genera un profilo a linee per confrontare
# positive, negative e negative-to-correction.

figure, axis = plt.subplots(
    figsize=(13, 6)
)


x_positions = np.arange(
    len(ALL_TASKS)
)


for model_name in LEGEND_MODELS:
    axis.plot(
        x_positions,
        legend_variants_comparison_df[
            model_name
        ],
        marker="o",
        linewidth=2,
        label=model_name,
    )


axis.set_title(
    "Confronto tra le tre varianti Legends"
)

axis.set_xlabel(
    "Task"
)

axis.set_ylabel(
    "Primary score"
)


axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    [
        TASK_DISPLAY_NAMES[
            task_name
        ]
        for task_name in ALL_TASKS
    ],
    rotation=30,
    ha="right",
)


axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    ZOOM_PLOTS_DIR
    / "legend_variants_lines.png",
)

In [ ]:
# CELLA 32 — Genera un grafico a barre raggruppate
# per confrontare i valori puntuali delle tre varianti Legends.

figure, axis = plt.subplots(
    figsize=(14, 6)
)


x_positions = np.arange(
    len(ALL_TASKS)
)

bar_width = (
    0.8
    / len(LEGEND_MODELS)
)


for model_index, model_name in enumerate(
    LEGEND_MODELS
):
    axis.bar(
        x_positions
        + model_index
        * bar_width
        - 0.4
        + bar_width / 2,
        legend_variants_comparison_df[
            model_name
        ],
        width=bar_width,
        label=model_name,
    )


axis.set_title(
    "Varianti Legends: confronto task per task"
)

axis.set_xlabel(
    "Task"
)

axis.set_ylabel(
    "Primary score"
)


axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    [
        TASK_DISPLAY_NAMES[
            task_name
        ]
        for task_name in ALL_TASKS
    ],
    rotation=30,
    ha="right",
)


axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    ZOOM_PLOTS_DIR
    / "legend_variants_grouped_bars.png",
)

In [ ]:
# CELLA 33 — Calcola punti complessivi, rank medio e numero di task
# vinti da ciascuna variante Legends nel confronto interno.

legend_ranking_summary_df = (
    task_rankings_automatic_df[
        task_rankings_automatic_df[
            "model_name"
        ]
        .isin(
            LEGEND_MODELS
        )
    ]
    .groupby(
        "model_name",
        as_index=False,
    )
    .agg(
        total_points=(
            "points",
            "sum",
        ),

        average_rank=(
            "average_rank",
            "mean",
        ),

        rank_standard_deviation=(
            "average_rank",
            "std",
        ),

        first_places_overall=(
            "rank",
            lambda values: int(
                (
                    values == 1
                )
                .sum()
            ),
        ),
    )
)


legend_internal_wins = {
    model_name: 0
    for model_name in LEGEND_MODELS
}


for task_name in ALL_TASKS:
    task_scores = (
        legend_variants_comparison_df
        .loc[
            task_name
        ]
    )

    maximum_score = float(
        task_scores.max()
    )

    winners = task_scores[
        np.isclose(
            task_scores,
            maximum_score,
            atol=RANKING_TOLERANCE,
            rtol=0,
        )
    ].index


    for winner in winners:
        legend_internal_wins[
            winner
        ] += 1


legend_ranking_summary_df[
    "internal_task_wins"
] = (
    legend_ranking_summary_df[
        "model_name"
    ]
    .map(
        legend_internal_wins
    )
)


legend_ranking_summary_df = (
    legend_ranking_summary_df
    .sort_values(
        [
            "total_points",
            "average_rank",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


display(
    legend_ranking_summary_df
    .round(4)
)

# Sezione 6 — Zoom 3: Regulation contro le varianti Legends

Questa sezione confronta il modello addestrato direttamente sulla normativa
con i tre modelli addestrati sulle rappresentazioni narrative.

Vengono prodotti:

- valori assoluti;
- profilo a linee;
- delta delle Legends rispetto a Regulation.

Un delta positivo indica che la variante Legends supera Regulation sul task.
Un delta negativo indica il contrario.

In [ ]:
# CELLA 35 — Costruisce la tabella dei valori assoluti
# e dei delta delle tre Legends rispetto a Regulation.

regulation_vs_legends_absolute_df = (
    all_primary_matrix_df[
        REGULATION_AND_LEGENDS_MODELS
    ]
    .copy()
)


regulation_vs_legends_delta_df = pd.DataFrame(
    index=ALL_TASKS
)


for legend_model in LEGEND_MODELS:
    regulation_vs_legends_delta_df[
        legend_model
    ] = (
        all_primary_matrix_df[
            legend_model
        ]
        - all_primary_matrix_df[
            "regulation"
        ]
    )


display(
    regulation_vs_legends_absolute_df
    .round(4)
)

display(
    regulation_vs_legends_delta_df
    .round(4)
)

In [ ]:
# CELLA 36 — Genera il profilo a linee di Regulation
# e delle tre varianti Legends.

figure, axis = plt.subplots(
    figsize=(13, 6)
)


x_positions = np.arange(
    len(ALL_TASKS)
)


for model_name in REGULATION_AND_LEGENDS_MODELS:
    axis.plot(
        x_positions,
        regulation_vs_legends_absolute_df[
            model_name
        ],
        marker="o",
        linewidth=2,
        label=model_name,
    )


axis.set_title(
    "Regulation contro rappresentazioni Legends"
)

axis.set_xlabel(
    "Task"
)

axis.set_ylabel(
    "Primary score"
)


axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    [
        TASK_DISPLAY_NAMES[
            task_name
        ]
        for task_name in ALL_TASKS
    ],
    rotation=30,
    ha="right",
)


axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    ZOOM_PLOTS_DIR
    / "regulation_vs_legends_lines.png",
)

In [ ]:
# CELLA 37 — Genera il grafico dei delta:
# score Legends meno score Regulation.

figure, axis = plt.subplots(
    figsize=(14, 6)
)


x_positions = np.arange(
    len(ALL_TASKS)
)

bar_width = (
    0.8
    / len(LEGEND_MODELS)
)


for model_index, model_name in enumerate(
    LEGEND_MODELS
):
    axis.bar(
        x_positions
        + model_index
        * bar_width
        - 0.4
        + bar_width / 2,
        regulation_vs_legends_delta_df[
            model_name
        ],
        width=bar_width,
        label=model_name,
    )


axis.axhline(
    0,
    linewidth=1,
)


axis.set_title(
    "Delta delle varianti Legends rispetto a Regulation"
)

axis.set_xlabel(
    "Task"
)

axis.set_ylabel(
    "Score Legends − score Regulation"
)


axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    [
        TASK_DISPLAY_NAMES[
            task_name
        ]
        for task_name in ALL_TASKS
    ],
    rotation=30,
    ha="right",
)


axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    ZOOM_PLOTS_DIR
    / "regulation_vs_legends_delta_bars.png",
)

# Sezione 7 — Generazione automatica: BLEU, ROUGE e BERTScore

Le metriche generative vengono mantenute separate.

## BLEU

Misura principalmente la sovrapposizione di sequenze di parole tra la risposta
generata e la reference.

Vengono mostrati:

- mean Sentence BLEU;
- Corpus BLEU.

## ROUGE

Misura quanto contenuto lessicale della reference viene recuperato.

Vengono mostrati:

- ROUGE-1 F1;
- ROUGE-2 F1;
- ROUGE-L F1.

## BERTScore

Confronta i token nel loro contesto semantico.

Vengono mostrati:

- BERTScore precision;
- BERTScore recall;
- BERTScore F1.

BERTScore F1 è utilizzato come metrica rappresentativa nel ranking generativo,
ma non sostituisce BLEU e ROUGE nell'analisi dettagliata.

In [ ]:
# CELLA 39 — Seleziona e ordina le principali metriche automatiche
# della generazione per i cinque modelli.

GENERATION_METRIC_COLUMNS = [
    "model_name",
    "mean_sentence_bleu",
    "corpus_bleu",
    "mean_rouge1_f1",
    "mean_rouge2_f1",
    "mean_rougeL_f1",
    "mean_bertscore_precision",
    "mean_bertscore_recall",
    "mean_bertscore_f1",
    "mean_reference_word_count",
    "mean_generated_word_count",
    "median_generated_word_count",
    "mean_word_length_ratio",
]


generation_metrics_table_df = (
    generation_by_model_df[
        [
            column_name
            for column_name in GENERATION_METRIC_COLUMNS
            if column_name
            in generation_by_model_df.columns
        ]
    ]
    .copy()
)


generation_metrics_table_df = sort_models(
    generation_metrics_table_df
)


display(
    generation_metrics_table_df
    .round(4)
)

In [ ]:
# CELLA 40 — Confronta mean Sentence BLEU e Corpus BLEU
# attraverso due barre per modello.

figure, axis = plt.subplots(
    figsize=(11, 6)
)


x_positions = np.arange(
    len(MODEL_ORDER)
)

bar_width = 0.35


axis.bar(
    x_positions
    - bar_width / 2,
    generation_metrics_table_df[
        "mean_sentence_bleu"
    ],
    width=bar_width,
    label="Mean Sentence BLEU",
)

axis.bar(
    x_positions
    + bar_width / 2,
    generation_metrics_table_df[
        "corpus_bleu"
    ],
    width=bar_width,
    label="Corpus BLEU",
)


axis.set_title(
    "Grounded generation — BLEU"
)

axis.set_xlabel(
    "Modello"
)

axis.set_ylabel(
    "BLEU score"
)


axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    MODEL_ORDER,
    rotation=25,
    ha="right",
)


axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    GENERATION_PLOTS_DIR
    / "generation_bleu.png",
)

In [ ]:
# CELLA 41 — Confronta ROUGE-1, ROUGE-2 e ROUGE-L F1
# con tre barre per modello.

figure, axis = plt.subplots(
    figsize=(12, 6)
)


x_positions = np.arange(
    len(MODEL_ORDER)
)

bar_width = 0.25


axis.bar(
    x_positions
    - bar_width,
    generation_metrics_table_df[
        "mean_rouge1_f1"
    ],
    width=bar_width,
    label="ROUGE-1 F1",
)

axis.bar(
    x_positions,
    generation_metrics_table_df[
        "mean_rouge2_f1"
    ],
    width=bar_width,
    label="ROUGE-2 F1",
)

axis.bar(
    x_positions
    + bar_width,
    generation_metrics_table_df[
        "mean_rougeL_f1"
    ],
    width=bar_width,
    label="ROUGE-L F1",
)


axis.set_title(
    "Grounded generation — ROUGE F1"
)

axis.set_xlabel(
    "Modello"
)

axis.set_ylabel(
    "ROUGE score"
)


axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    MODEL_ORDER,
    rotation=25,
    ha="right",
)


axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    GENERATION_PLOTS_DIR
    / "generation_rouge_f1.png",
)

In [ ]:
# CELLA 42 — Confronta precision, recall e F1 di BERTScore
# con tre barre per modello.

figure, axis = plt.subplots(
    figsize=(12, 6)
)


x_positions = np.arange(
    len(MODEL_ORDER)
)

bar_width = 0.25


axis.bar(
    x_positions
    - bar_width,
    generation_metrics_table_df[
        "mean_bertscore_precision"
    ],
    width=bar_width,
    label="BERTScore precision",
)

axis.bar(
    x_positions,
    generation_metrics_table_df[
        "mean_bertscore_recall"
    ],
    width=bar_width,
    label="BERTScore recall",
)

axis.bar(
    x_positions
    + bar_width,
    generation_metrics_table_df[
        "mean_bertscore_f1"
    ],
    width=bar_width,
    label="BERTScore F1",
)


axis.set_title(
    "Grounded generation — BERTScore"
)

axis.set_xlabel(
    "Modello"
)

axis.set_ylabel(
    "BERTScore"
)


axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    MODEL_ORDER,
    rotation=25,
    ha="right",
)


axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    GENERATION_PLOTS_DIR
    / "generation_bertscore.png",
)

In [ ]:
# CELLA 43 — Costruisce una heatmap con BLEU, ROUGE e BERTScore
# mantenendo visibili i valori originali.

generation_heatmap_columns = [
    "corpus_bleu",
    "mean_rouge1_f1",
    "mean_rouge2_f1",
    "mean_rougeL_f1",
    "mean_bertscore_f1",
]


generation_heatmap_df = (
    generation_metrics_table_df
    .set_index("model_name")
    [
        generation_heatmap_columns
    ]
)


generation_heatmap_values = (
    generation_heatmap_df
    .to_numpy(
        dtype=float
    )
)


figure, axis = plt.subplots(
    figsize=(9, 5)
)

image = axis.imshow(
    generation_heatmap_values,
    aspect="auto",
)


axis.set_title(
    "Metriche automatiche della generazione"
)

axis.set_xlabel(
    "Metrica"
)

axis.set_ylabel(
    "Modello"
)


axis.set_xticks(
    np.arange(
        len(generation_heatmap_columns)
    )
)

axis.set_xticklabels(
    [
        "Corpus BLEU",
        "ROUGE-1 F1",
        "ROUGE-2 F1",
        "ROUGE-L F1",
        "BERTScore F1",
    ],
    rotation=30,
    ha="right",
)


axis.set_yticks(
    np.arange(
        len(MODEL_ORDER)
    )
)

axis.set_yticklabels(
    MODEL_ORDER
)


annotate_heatmap(
    axis,
    generation_heatmap_values,
    decimals=3,
)


figure.colorbar(
    image,
    ax=axis,
    label="Score originale",
)


save_figure(
    figure,
    GENERATION_PLOTS_DIR
    / "generation_metrics_heatmap_raw.png",
)

In [ ]:
# CELLA 44 — Normalizza ogni metrica generativa tra 0 e 1
# esclusivamente per visualizzare la posizione relativa dei modelli.

generation_normalized_df = (
    generation_heatmap_df
    .copy()
)


for column_name in generation_normalized_df.columns:
    column_minimum = float(
        generation_normalized_df[
            column_name
        ]
        .min()
    )

    column_maximum = float(
        generation_normalized_df[
            column_name
        ]
        .max()
    )

    if np.isclose(
        column_minimum,
        column_maximum,
    ):
        generation_normalized_df[
            column_name
        ] = 1.0

    else:
        generation_normalized_df[
            column_name
        ] = (
            generation_normalized_df[
                column_name
            ]
            - column_minimum
        ) / (
            column_maximum
            - column_minimum
        )


normalized_values = (
    generation_normalized_df
    .to_numpy(
        dtype=float
    )
)


figure, axis = plt.subplots(
    figsize=(9, 5)
)

image = axis.imshow(
    normalized_values,
    aspect="auto",
)


axis.set_title(
    "Metriche generative normalizzate internamente"
)

axis.set_xlabel(
    "Metrica"
)

axis.set_ylabel(
    "Modello"
)


axis.set_xticks(
    np.arange(
        len(generation_heatmap_columns)
    )
)

axis.set_xticklabels(
    [
        "Corpus BLEU",
        "ROUGE-1 F1",
        "ROUGE-2 F1",
        "ROUGE-L F1",
        "BERTScore F1",
    ],
    rotation=30,
    ha="right",
)


axis.set_yticks(
    np.arange(
        len(MODEL_ORDER)
    )
)

axis.set_yticklabels(
    MODEL_ORDER
)


annotate_heatmap(
    axis,
    normalized_values,
    decimals=2,
)


figure.colorbar(
    image,
    ax=axis,
    label="Posizione relativa 0–1",
)


save_figure(
    figure,
    GENERATION_PLOTS_DIR
    / "generation_metrics_heatmap_normalized.png",
)

In [ ]:
# CELLA 45 — Confronta la lunghezza media delle risposte generate
# con quella delle reference e mostra il rapporto tra le due.

figure, axis = plt.subplots(
    figsize=(11, 5)
)


x_positions = np.arange(
    len(MODEL_ORDER)
)

bar_width = 0.35


axis.bar(
    x_positions
    - bar_width / 2,
    generation_metrics_table_df[
        "mean_reference_word_count"
    ],
    width=bar_width,
    label="Reference",
)

axis.bar(
    x_positions
    + bar_width / 2,
    generation_metrics_table_df[
        "mean_generated_word_count"
    ],
    width=bar_width,
    label="Generated",
)


axis.set_title(
    "Lunghezza media delle risposte"
)

axis.set_xlabel(
    "Modello"
)

axis.set_ylabel(
    "Numero medio di parole"
)


axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    MODEL_ORDER,
    rotation=25,
    ha="right",
)


axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    GENERATION_PLOTS_DIR
    / "generation_word_count.png",
)


figure, axis = plt.subplots(
    figsize=(10, 5)
)


axis.bar(
    MODEL_ORDER,
    generation_metrics_table_df[
        "mean_word_length_ratio"
    ],
)


axis.axhline(
    1,
    linewidth=1,
)


axis.set_title(
    "Rapporto tra lunghezza generata e reference"
)

axis.set_xlabel(
    "Modello"
)

axis.set_ylabel(
    "Generated words / reference words"
)

axis.tick_params(
    axis="x",
    rotation=25,
)


apply_y_grid(
    axis
)


save_figure(
    figure,
    GENERATION_PLOTS_DIR
    / "generation_word_length_ratio.png",
)

# Sezione 8 — Valutazione soggettiva della generazione

Il notebook cerca:

1. `subjective_evaluation_completed.xlsx`;
2. in alternativa `subjective_evaluation_to_complete.xlsx`.

La sezione viene eseguita solo se tutte le 100 risposte contengono:

- `valutazione_A`;
- `valutazione_B`.

Per ogni risposta viene calcolata:

```text
subjective_mean_score =
(valutazione_A + valutazione_B) / 2

In [ ]:


# Cella 47 — Caricamento e validazione dell’Excel
# CELLA 47 — Cerca il file soggettivo, legge il foglio corretto
# e verifica che le valutazioni siano intere e comprese tra 0 e 5.

subjective_file_path = next(
    (
        path
        for path in SUBJECTIVE_FILE_CANDIDATES
        if path.exists()
    ),
    None,
)


subjective_available = False
subjective_evaluation_df = pd.DataFrame()
subjective_status_df = pd.DataFrame()


if subjective_file_path is None:
    warnings.warn(
        "Nessun file di valutazione soggettiva trovato."
    )

else:
    excel_file = pd.ExcelFile(
        subjective_file_path
    )

    subjective_sheet_name = (
        "Valutazioni"
        if "Valutazioni"
        in excel_file.sheet_names
        else excel_file.sheet_names[0]
    )


    subjective_evaluation_df = pd.read_excel(
        subjective_file_path,
        sheet_name=subjective_sheet_name,
    )


    required_columns = {
        "model_name",
        "example_id",
        "valutazione_A",
        "valutazione_B",
    }


    missing_columns = (
        required_columns
        - set(
            subjective_evaluation_df.columns
        )
    )


    if missing_columns:
        raise RuntimeError(
            "Nell'Excel mancano le colonne: "
            f"{sorted(missing_columns)}"
        )


    for column_name in [
        "valutazione_A",
        "valutazione_B",
    ]:
        subjective_evaluation_df[
            column_name
        ] = pd.to_numeric(
            subjective_evaluation_df[
                column_name
            ],
            errors="coerce",
        )


    completed_both = int(
        (
            subjective_evaluation_df[
                "valutazione_A"
            ]
            .notna()
            &
            subjective_evaluation_df[
                "valutazione_B"
            ]
            .notna()
        )
        .sum()
    )


    available_scores = pd.concat(
        [
            subjective_evaluation_df[
                "valutazione_A"
            ],

            subjective_evaluation_df[
                "valutazione_B"
            ],
        ],
        ignore_index=True,
    ).dropna()


    if (
        (
            available_scores < 0
        )
        |
        (
            available_scores > 5
        )
    ).any():
        raise RuntimeError(
            "Sono presenti valutazioni fuori dall'intervallo 0–5."
        )


    if (
        available_scores
        % 1
        != 0
    ).any():
        raise RuntimeError(
            "Le valutazioni A e B devono essere intere."
        )


    subjective_available = (
        len(subjective_evaluation_df)
        == EXPECTED_GENERATION_ROWS
        and completed_both
        == EXPECTED_GENERATION_ROWS
    )


    subjective_status_df = pd.DataFrame([
        {
            "file": str(
                subjective_file_path
            ),

            "sheet": (
                subjective_sheet_name
            ),

            "total_rows": len(
                subjective_evaluation_df
            ),

            "completed_both": (
                completed_both
            ),

            "expected_rows": (
                EXPECTED_GENERATION_ROWS
            ),

            "subjective_available": (
                subjective_available
            ),
        }
    ])


    if not subjective_available:
        warnings.warn(
            "Valutazione soggettiva incompleta: "
            "le relative analisi saranno saltate."
        )


display(
    subjective_status_df
)

In [ ]:
# CELLA 48 — Calcola i punteggi soggettivi per risposta,
# le statistiche per modello e l'accordo tra valutatori.

subjective_by_example_df = pd.DataFrame()
subjective_by_model_df = pd.DataFrame()
subjective_agreement_df = pd.DataFrame()


if subjective_available:
    subjective_by_example_df = (
        subjective_evaluation_df
        .copy()
    )


    subjective_by_example_df[
        "subjective_mean_score"
    ] = (
        subjective_by_example_df[
            "valutazione_A"
        ]
        +
        subjective_by_example_df[
            "valutazione_B"
        ]
    ) / 2


    subjective_by_example_df[
        "evaluator_absolute_difference"
    ] = (
        subjective_by_example_df[
            "valutazione_A"
        ]
        -
        subjective_by_example_df[
            "valutazione_B"
        ]
    ).abs()


    subjective_by_model_df = (
        subjective_by_example_df
        .groupby(
            "model_name",
            as_index=False,
        )
        .agg(
            n_answers=(
                "example_id",
                "count",
            ),

            mean_evaluation_A=(
                "valutazione_A",
                "mean",
            ),

            mean_evaluation_B=(
                "valutazione_B",
                "mean",
            ),

            mean_subjective_score=(
                "subjective_mean_score",
                "mean",
            ),

            median_subjective_score=(
                "subjective_mean_score",
                "median",
            ),

            std_subjective_score=(
                "subjective_mean_score",
                "std",
            ),

            exact_agreement_rate=(
                "evaluator_absolute_difference",
                lambda values: float(
                    (
                        values == 0
                    )
                    .mean()
                ),
            ),

            within_one_point_rate=(
                "evaluator_absolute_difference",
                lambda values: float(
                    (
                        values <= 1
                    )
                    .mean()
                ),
            ),

            mean_absolute_difference=(
                "evaluator_absolute_difference",
                "mean",
            ),
        )
    )


    subjective_by_model_df = sort_models(
        subjective_by_model_df
    )


    weighted_kappa = float(
        cohen_kappa_score(
            subjective_by_example_df[
                "valutazione_A"
            ],
            subjective_by_example_df[
                "valutazione_B"
            ],
            weights="quadratic",
        )
    )


    subjective_agreement_df = pd.DataFrame([
        {
            "n_answers": len(
                subjective_by_example_df
            ),

            "exact_agreement_rate": float(
                (
                    subjective_by_example_df[
                        "evaluator_absolute_difference"
                    ]
                    == 0
                )
                .mean()
            ),

            "within_one_point_rate": float(
                (
                    subjective_by_example_df[
                        "evaluator_absolute_difference"
                    ]
                    <= 1
                )
                .mean()
            ),

            "mean_absolute_difference": float(
                subjective_by_example_df[
                    "evaluator_absolute_difference"
                ]
                .mean()
            ),

            "quadratic_weighted_kappa": (
                weighted_kappa
            ),
        }
    ])


    display(
        subjective_by_model_df
        .round(4)
    )

    display(
        subjective_agreement_df
        .round(4)
    )

In [ ]:
# CELLA 49 — Costruisce il ranking soggettivo della generazione
# utilizzando la media tra valutatore A e valutatore B.

subjective_generation_ranking_df = pd.DataFrame()


if subjective_available:
    subjective_ranking_input_df = (
        subjective_by_model_df[
            [
                "model_name",
                "mean_subjective_score",
            ]
        ]
        .rename(
            columns={
                "mean_subjective_score": (
                    "primary_score"
                )
            }
        )
    )


    subjective_ranking_input_df[
        "task_name"
    ] = GENERATION_TASK

    subjective_ranking_input_df[
        "primary_metric_name"
    ] = "mean_subjective_score"


    subjective_generation_ranking_df = (
        rank_models_for_task(
            subjective_ranking_input_df
        )
    )


    display(
        subjective_generation_ranking_df
        .round(4)
    )

In [ ]:
# CELLA 50 — Visualizza le medie A e B e il ranking
# soggettivo complessivo della generazione.

if subjective_available:
    figure, axis = plt.subplots(
        figsize=(11, 5)
    )


    x_positions = np.arange(
        len(MODEL_ORDER)
    )

    bar_width = 0.35


    axis.bar(
        x_positions
        - bar_width / 2,
        subjective_by_model_df[
            "mean_evaluation_A"
        ],
        width=bar_width,
        label="Valutatore A",
    )

    axis.bar(
        x_positions
        + bar_width / 2,
        subjective_by_model_df[
            "mean_evaluation_B"
        ],
        width=bar_width,
        label="Valutatore B",
    )


    axis.set_title(
        "Valutazione soggettiva media A e B"
    )

    axis.set_xlabel(
        "Modello"
    )

    axis.set_ylabel(
        "Punteggio medio 0–5"
    )


    axis.set_xticks(
        x_positions
    )

    axis.set_xticklabels(
        MODEL_ORDER,
        rotation=25,
        ha="right",
    )


    axis.set_ylim(
        0,
        5,
    )

    axis.legend()

    apply_y_grid(
        axis
    )


    save_figure(
        figure,
        SUBJECTIVE_PLOTS_DIR
        / "subjective_evaluation_A_B.png",
    )


    subjective_rank_plot_df = (
        subjective_generation_ranking_df
        .sort_values(
            "primary_score",
            ascending=True,
        )
    )


    figure, axis = plt.subplots(
        figsize=(9, 5)
    )


    axis.barh(
        subjective_rank_plot_df[
            "model_name"
        ],
        subjective_rank_plot_df[
            "primary_score"
        ],
    )


    axis.set_title(
        "Ranking soggettivo della generazione"
    )

    axis.set_xlabel(
        "Valutazione media 0–5"
    )

    axis.set_ylabel(
        "Modello"
    )

    axis.set_xlim(
        0,
        5,
    )


    apply_x_grid(
        axis
    )


    save_figure(
        figure,
        SUBJECTIVE_PLOTS_DIR
        / "subjective_generation_ranking.png",
    )

In [ ]:
# CELLA 51 — Visualizza la distribuzione dei punteggi medi
# e il boxplot delle valutazioni per modello.

if subjective_available:
    subjective_distribution_df = (
        subjective_by_example_df
        .groupby(
            [
                "model_name",
                "subjective_mean_score",
            ]
        )
        .size()
        .reset_index(
            name="count"
        )
    )


    subjective_distribution_pivot_df = (
        subjective_distribution_df
        .pivot(
            index="model_name",
            columns="subjective_mean_score",
            values="count",
        )
        .fillna(0)
        .reindex(
            MODEL_ORDER
        )
    )


    figure, axis = plt.subplots(
        figsize=(11, 6)
    )


    bottom_values = np.zeros(
        len(MODEL_ORDER)
    )


    for score_value in (
        subjective_distribution_pivot_df
        .columns
    ):
        values = (
            subjective_distribution_pivot_df[
                score_value
            ]
            .to_numpy(
                dtype=float
            )
        )

        axis.bar(
            MODEL_ORDER,
            values,
            bottom=bottom_values,
            label=str(score_value),
        )

        bottom_values += values


    axis.set_title(
        "Distribuzione delle valutazioni soggettive"
    )

    axis.set_xlabel(
        "Modello"
    )

    axis.set_ylabel(
        "Numero di risposte"
    )

    axis.tick_params(
        axis="x",
        rotation=25,
    )

    axis.legend(
        title="Punteggio medio"
    )

    apply_y_grid(
        axis
    )


    save_figure(
        figure,
        SUBJECTIVE_PLOTS_DIR
        / "subjective_score_distribution.png",
    )


    boxplot_data = [
        subjective_by_example_df[
            subjective_by_example_df[
                "model_name"
            ]
            == model_name
        ][
            "subjective_mean_score"
        ]
        .to_numpy()
        for model_name in MODEL_ORDER
    ]


    figure, axis = plt.subplots(
        figsize=(11, 6)
    )


    axis.boxplot(
        boxplot_data,
        tick_labels=MODEL_ORDER,
    )


    axis.set_title(
        "Distribuzione delle valutazioni soggettive per modello"
    )

    axis.set_xlabel(
        "Modello"
    )

    axis.set_ylabel(
        "Punteggio soggettivo medio"
    )

    axis.tick_params(
        axis="x",
        rotation=25,
    )

    axis.set_ylim(
        0,
        5,
    )

    apply_y_grid(
        axis
    )


    save_figure(
        figure,
        SUBJECTIVE_PLOTS_DIR
        / "subjective_score_boxplot.png",
    )

In [ ]:
# CELLA 52 — Confronta BERTScore F1 medio e punteggio soggettivo
# medio attraverso uno scatter plot con un punto per modello.

automatic_vs_subjective_by_model_df = pd.DataFrame()


if subjective_available:
    automatic_generation_df = (
        generation_by_model_df[
            [
                "model_name",
                "mean_bertscore_f1",
            ]
        ]
    )


    automatic_vs_subjective_by_model_df = (
        automatic_generation_df
        .merge(
            subjective_by_model_df[
                [
                    "model_name",
                    "mean_subjective_score",
                ]
            ],
            on="model_name",
            validate="one_to_one",
        )
    )


    model_level_spearman = spearmanr(
        automatic_vs_subjective_by_model_df[
            "mean_bertscore_f1"
        ],
        automatic_vs_subjective_by_model_df[
            "mean_subjective_score"
        ],
    )


    automatic_vs_subjective_by_model_df[
        "spearman_correlation"
    ] = float(
        model_level_spearman.statistic
    )


    figure, axis = plt.subplots(
        figsize=(8, 6)
    )


    axis.scatter(
        automatic_vs_subjective_by_model_df[
            "mean_bertscore_f1"
        ],
        automatic_vs_subjective_by_model_df[
            "mean_subjective_score"
        ],
    )


    for row in (
        automatic_vs_subjective_by_model_df
        .itertuples(
            index=False
        )
    ):
        axis.annotate(
            row.model_name,
            (
                row.mean_bertscore_f1,
                row.mean_subjective_score,
            ),
        )


    axis.set_title(
        "BERTScore e valutazione soggettiva per modello"
    )

    axis.set_xlabel(
        "BERTScore F1 medio"
    )

    axis.set_ylabel(
        "Valutazione soggettiva media"
    )


    apply_y_grid(
        axis
    )


    save_figure(
        figure,
        SUBJECTIVE_PLOTS_DIR
        / "automatic_vs_subjective_by_model.png",
    )


    display(
        automatic_vs_subjective_by_model_df
        .round(4)
    )

In [ ]:
# CELLA 53 — Confronta BERTScore F1 e valutazione soggettiva
# sulle 100 risposte generative, distinguendo i modelli.

automatic_vs_subjective_by_example_df = pd.DataFrame()


if subjective_available:
    automatic_vs_subjective_by_example_df = (
        generation_by_example_df[
            [
                "model_name",
                "example_id",
                "bertscore_f1",
            ]
        ]
        .merge(
            subjective_by_example_df[
                [
                    "model_name",
                    "example_id",
                    "subjective_mean_score",
                ]
            ],
            on=[
                "model_name",
                "example_id",
            ],
            validate="one_to_one",
        )
    )


    example_level_spearman = spearmanr(
        automatic_vs_subjective_by_example_df[
            "bertscore_f1"
        ],
        automatic_vs_subjective_by_example_df[
            "subjective_mean_score"
        ],
    )


    automatic_vs_subjective_by_example_df[
        "overall_spearman_correlation"
    ] = float(
        example_level_spearman.statistic
    )


    figure, axis = plt.subplots(
        figsize=(9, 6)
    )


    for model_name in MODEL_ORDER:
        model_data = (
            automatic_vs_subjective_by_example_df[
                automatic_vs_subjective_by_example_df[
                    "model_name"
                ]
                == model_name
            ]
        )

        axis.scatter(
            model_data[
                "bertscore_f1"
            ],
            model_data[
                "subjective_mean_score"
            ],
            label=model_name,
            alpha=0.75,
        )


    axis.set_title(
        "BERTScore e valutazione soggettiva per risposta"
    )

    axis.set_xlabel(
        "BERTScore F1"
    )

    axis.set_ylabel(
        "Valutazione soggettiva media"
    )

    axis.legend()

    apply_y_grid(
        axis
    )


    save_figure(
        figure,
        SUBJECTIVE_PLOTS_DIR
        / "automatic_vs_subjective_by_example.png",
    )

In [ ]:
# CELLA 55 — Definisce la funzione che aggrega i punti
# e calcola statistiche sui piazzamenti.

def aggregate_task_rankings(
    task_rankings: pd.DataFrame,
    included_tasks: list[str],
    ranking_name: str,
) -> pd.DataFrame:
    """Aggrega punti e rank su un insieme selezionato di task."""

    selected = task_rankings[
        task_rankings[
            "task_name"
        ]
        .isin(
            included_tasks
        )
    ].copy()


    aggregated = (
        selected
        .groupby(
            "model_name",
            as_index=False,
        )
        .agg(
            total_points=(
                "points",
                "sum",
            ),

            average_points=(
                "points",
                "mean",
            ),

            average_rank=(
                "average_rank",
                "mean",
            ),

            median_rank=(
                "average_rank",
                "median",
            ),

            best_rank=(
                "rank",
                "min",
            ),

            worst_rank=(
                "rank_end",
                "max",
            ),

            first_places=(
                "rank",
                lambda values: int(
                    (
                        values == 1
                    )
                    .sum()
                ),
            ),

            top_2_finishes=(
                "rank",
                lambda values: int(
                    (
                        values <= 2
                    )
                    .sum()
                ),
            ),

            last_places=(
                "rank_end",
                lambda values: int(
                    (
                        values
                        == len(MODEL_ORDER)
                    )
                    .sum()
                ),
            ),

            n_tasks=(
                "task_name",
                "nunique",
            ),
        )
    )


    aggregated[
        "ranking_name"
    ] = ranking_name


    aggregated = (
        aggregated
        .sort_values(
            [
                "total_points",
                "average_rank",
                "model_name",
            ],
            ascending=[
                False,
                True,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )


    aggregated[
        "overall_position"
    ] = np.arange(
        1,
        len(aggregated) + 1,
    )


    return aggregated

In [ ]:
# CELLA 56 — Calcola il ranking sui sei task classificatori
# e il ranking complessivo automatico sugli otto task.

classification_ranking_df = (
    aggregate_task_rankings(
        task_rankings=(
            task_rankings_automatic_df
        ),
        included_tasks=(
            CLASSIFICATION_TASKS
        ),
        ranking_name=(
            "classification_ranking"
        ),
    )
)


overall_ranking_automatic_df = (
    aggregate_task_rankings(
        task_rankings=(
            task_rankings_automatic_df
        ),
        included_tasks=(
            ALL_TASKS
        ),
        ranking_name=(
            "overall_automatic"
        ),
    )
)


display(
    classification_ranking_df
    .round(4)
)

display(
    overall_ranking_automatic_df
    .round(4)
)

In [ ]:
# CELLA 57 — Quando disponibile, sostituisce il ranking generativo
# automatico con quello soggettivo e ricalcola il ranking complessivo.

task_rankings_subjective_scenario_df = pd.DataFrame()
overall_ranking_subjective_scenario_df = pd.DataFrame()


if subjective_available:
    non_generation_rankings_df = (
        task_rankings_automatic_df[
            task_rankings_automatic_df[
                "task_name"
            ]
            != GENERATION_TASK
        ]
        .copy()
    )


    task_rankings_subjective_scenario_df = (
        pd.concat(
            [
                non_generation_rankings_df,
                subjective_generation_ranking_df,
            ],
            ignore_index=True,
            sort=False,
        )
    )


    overall_ranking_subjective_scenario_df = (
        aggregate_task_rankings(
            task_rankings=(
                task_rankings_subjective_scenario_df
            ),
            included_tasks=(
                ALL_TASKS
            ),
            ranking_name=(
                "overall_subjective_generation_scenario"
            ),
        )
    )


    display(
        overall_ranking_subjective_scenario_df
        .round(4)
    )

In [ ]:
# CELLA 58 — Costruisce e visualizza la matrice dei punti
# ottenuti dai modelli nei singoli task.

task_points_matrix_df = (
    task_rankings_automatic_df
    .pivot(
        index="model_name",
        columns="task_name",
        values="points",
    )
    .reindex(
        index=MODEL_ORDER,
        columns=ALL_TASKS,
    )
)


points_matrix = (
    task_points_matrix_df
    .to_numpy(
        dtype=float
    )
)


figure, axis = plt.subplots(
    figsize=(13, 5)
)

image = axis.imshow(
    points_matrix,
    aspect="auto",
)


axis.set_title(
    "Punti ottenuti dai modelli nei singoli task"
)

axis.set_xlabel(
    "Task"
)

axis.set_ylabel(
    "Modello"
)


axis.set_xticks(
    np.arange(
        len(ALL_TASKS)
    )
)

axis.set_xticklabels(
    [
        TASK_DISPLAY_NAMES[
            task_name
        ]
        for task_name in ALL_TASKS
    ],
    rotation=30,
    ha="right",
)


axis.set_yticks(
    np.arange(
        len(MODEL_ORDER)
    )
)

axis.set_yticklabels(
    MODEL_ORDER
)


annotate_heatmap(
    axis,
    points_matrix,
    decimals=1,
)


figure.colorbar(
    image,
    ax=axis,
    label="Punti",
)


save_figure(
    figure,
    RANKING_PLOTS_DIR
    / "task_points_heatmap.png",
)

In [ ]:
# CELLA 59 — Genera un line plot delle posizioni ottenute
# dai modelli nei diversi task. Il primo posto è visualizzato in alto.

rank_matrix_df = (
    task_rankings_automatic_df
    .pivot(
        index="model_name",
        columns="task_name",
        values="average_rank",
    )
    .reindex(
        index=MODEL_ORDER,
        columns=ALL_TASKS,
    )
)


figure, axis = plt.subplots(
    figsize=(13, 6)
)


x_positions = np.arange(
    len(ALL_TASKS)
)


for model_name in MODEL_ORDER:
    axis.plot(
        x_positions,
        rank_matrix_df.loc[
            model_name
        ],
        marker="o",
        linewidth=2,
        label=model_name,
    )


axis.set_title(
    "Profilo dei piazzamenti per task"
)

axis.set_xlabel(
    "Task"
)

axis.set_ylabel(
    "Posizione"
)


axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    [
        TASK_DISPLAY_NAMES[
            task_name
        ]
        for task_name in ALL_TASKS
    ],
    rotation=30,
    ha="right",
)


axis.set_yticks(
    [
        1,
        2,
        3,
        4,
        5,
    ]
)

axis.invert_yaxis()

axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    RANKING_PLOTS_DIR
    / "ranking_profiles_lines.png",
)

In [ ]:
# CELLA 60 — Genera i grafici orizzontali del ranking
# classificatorio, automatico complessivo e soggettivo alternativo.

def plot_ranking_bars(
    ranking_dataframe: pd.DataFrame,
    title: str,
    output_path: Path,
):
    """Crea un grafico orizzontale ordinato per punti."""

    plot_data = (
        ranking_dataframe
        .sort_values(
            "total_points",
            ascending=True,
        )
    )


    figure, axis = plt.subplots(
        figsize=(9, 5)
    )


    axis.barh(
        plot_data[
            "model_name"
        ],
        plot_data[
            "total_points"
        ],
    )


    axis.set_title(
        title
    )

    axis.set_xlabel(
        "Punti complessivi"
    )

    axis.set_ylabel(
        "Modello"
    )


    for row in plot_data.itertuples(
        index=False
    ):
        axis.text(
            row.total_points,
            row.model_name,
            f" {row.total_points:.1f}",
            va="center",
        )


    apply_x_grid(
        axis
    )


    save_figure(
        figure,
        output_path,
    )


plot_ranking_bars(
    classification_ranking_df,
    "Ranking aggregato dei task classificatori",
    RANKING_PLOTS_DIR
    / "classification_ranking.png",
)


plot_ranking_bars(
    overall_ranking_automatic_df,
    "Ranking complessivo automatico",
    RANKING_PLOTS_DIR
    / "overall_ranking_automatic.png",
)


if subjective_available:
    plot_ranking_bars(
        overall_ranking_subjective_scenario_df,
        "Ranking complessivo con generazione soggettiva",
        RANKING_PLOTS_DIR
        / "overall_ranking_subjective.png",
    )

In [ ]:
# CELLA 61 — Calcola rank medio, deviazione standard,
# primi posti, top-2 e ultimi posti per ciascun modello.

ranking_stability_df = (
    task_rankings_automatic_df
    .groupby(
        "model_name",
        as_index=False,
    )
    .agg(
        average_rank=(
            "average_rank",
            "mean",
        ),

        median_rank=(
            "average_rank",
            "median",
        ),

        rank_standard_deviation=(
            "average_rank",
            "std",
        ),

        first_places=(
            "rank",
            lambda values: int(
                (
                    values == 1
                )
                .sum()
            ),
        ),

        top_2_finishes=(
            "rank",
            lambda values: int(
                (
                    values <= 2
                )
                .sum()
            ),
        ),

        last_places=(
            "rank_end",
            lambda values: int(
                (
                    values
                    == len(MODEL_ORDER)
                )
                .sum()
            ),
        ),
    )
)


ranking_stability_df = sort_models(
    ranking_stability_df
)


display(
    ranking_stability_df
    .round(4)
)


figure, axis = plt.subplots(
    figsize=(10, 6)
)


axis.errorbar(
    ranking_stability_df[
        "model_name"
    ],
    ranking_stability_df[
        "average_rank"
    ],
    yerr=ranking_stability_df[
        "rank_standard_deviation"
    ],
    fmt="o",
    capsize=5,
)


axis.set_title(
    "Rank medio e variabilità dei piazzamenti"
)

axis.set_xlabel(
    "Modello"
)

axis.set_ylabel(
    "Rank medio ± deviazione standard"
)

axis.tick_params(
    axis="x",
    rotation=25,
)

axis.invert_yaxis()

apply_y_grid(
    axis
)


save_figure(
    figure,
    RANKING_PLOTS_DIR
    / "ranking_stability_errorbars.png",
)

In [ ]:
# CELLA 62 — Costruisce la matrice che conta su quanti task
# il modello della riga supera il modello della colonna.

dominance_matrix_df = pd.DataFrame(
    0,
    index=MODEL_ORDER,
    columns=MODEL_ORDER,
    dtype=int,
)

tie_matrix_df = pd.DataFrame(
    0,
    index=MODEL_ORDER,
    columns=MODEL_ORDER,
    dtype=int,
)


for task_name in ALL_TASKS:
    task_scores = (
        automatic_primary_scores_df[
            automatic_primary_scores_df[
                "task_name"
            ]
            == task_name
        ]
        .set_index("model_name")
        ["primary_score"]
    )


    for model_a in MODEL_ORDER:
        for model_b in MODEL_ORDER:
            if model_a == model_b:
                continue

            score_a = float(
                task_scores.loc[
                    model_a
                ]
            )

            score_b = float(
                task_scores.loc[
                    model_b
                ]
            )


            if np.isclose(
                score_a,
                score_b,
                atol=RANKING_TOLERANCE,
                rtol=0,
            ):
                tie_matrix_df.loc[
                    model_a,
                    model_b,
                ] += 1

            elif score_a > score_b:
                dominance_matrix_df.loc[
                    model_a,
                    model_b,
                ] += 1


dominance_values = (
    dominance_matrix_df
    .to_numpy(
        dtype=float
    )
)


figure, axis = plt.subplots(
    figsize=(8, 7)
)

image = axis.imshow(
    dominance_values,
    aspect="equal",
)


axis.set_title(
    "Matrice di dominanza task-level"
)

axis.set_xlabel(
    "Modello confrontato"
)

axis.set_ylabel(
    "Modello della riga"
)


axis.set_xticks(
    np.arange(
        len(MODEL_ORDER)
    )
)

axis.set_xticklabels(
    MODEL_ORDER,
    rotation=30,
    ha="right",
)


axis.set_yticks(
    np.arange(
        len(MODEL_ORDER)
    )
)

axis.set_yticklabels(
    MODEL_ORDER
)


annotate_heatmap(
    axis,
    dominance_values,
    decimals=0,
)


figure.colorbar(
    image,
    ax=axis,
    label="Task vinti",
)


save_figure(
    figure,
    RANKING_PLOTS_DIR
    / "model_dominance_matrix.png",
)

# Sezione 10 — Approfondimenti diagnostici

Questa sezione mantiene alcune analisi utili già presenti nel notebook
precedente:

- confusion matrix;
- variazioni dei tipi di errore rispetto alla baseline;
- distribuzioni STS-B;
- distanza dal valore corretto;
- MAE e RMSE.

Questi grafici sono principalmente destinati all'appendice o a slide
specifiche dedicate ai casi più interessanti.

In [ ]:
# CELLA 64 — Genera una confusion matrix per ciascuna combinazione
# modello-task classificatorio utilizzando le predizioni già calcolate.

for task_name in CLASSIFICATION_TASKS:
    labels = TASK_LABELS[
        task_name
    ]


    for model_name in MODEL_ORDER:
        model_task_data = (
            classification_results_df[
                (
                    classification_results_df[
                        "task_name"
                    ]
                    == task_name
                )
                &
                (
                    classification_results_df[
                        "model_name"
                    ]
                    == model_name
                )
            ]
        )


        confusion_matrix = (
            pd.crosstab(
                model_task_data[
                    "y_true"
                ],
                model_task_data[
                    "y_pred"
                ],
            )
            .reindex(
                index=labels,
                columns=labels,
                fill_value=0,
            )
        )


        matrix_values = (
            confusion_matrix
            .to_numpy(
                dtype=float
            )
        )


        figure, axis = plt.subplots(
            figsize=(6, 5)
        )

        image = axis.imshow(
            matrix_values,
            aspect="equal",
        )


        axis.set_title(
            f"{TASK_DISPLAY_NAMES[task_name]} — {model_name}"
        )

        axis.set_xlabel(
            "Predicted label"
        )

        axis.set_ylabel(
            "True label"
        )


        axis.set_xticks(
            np.arange(
                len(labels)
            )
        )

        axis.set_xticklabels(
            labels,
            rotation=30,
            ha="right",
        )


        axis.set_yticks(
            np.arange(
                len(labels)
            )
        )

        axis.set_yticklabels(
            labels
        )


        annotate_heatmap(
            axis,
            matrix_values,
            decimals=0,
        )


        figure.colorbar(
            image,
            ax=axis,
            label="Numero di esempi",
        )


        save_figure(
            figure,
            CONFUSION_MATRIX_DIR
            / f"{task_name}_{model_name}.png",
        )

In [ ]:
# CELLA 65 — Genera un grafico per ciascun task con la riduzione
# o l'aumento dei singoli tipi di errore rispetto alla baseline.

for task_name in CLASSIFICATION_TASKS:
    task_error_data = (
        classification_error_adapter_vs_base_df[
            classification_error_adapter_vs_base_df[
                "task_name"
            ]
            == task_name
        ]
        .copy()
    )


    if task_error_data.empty:
        continue


    task_error_data[
        "error_transition"
    ] = (
        task_error_data[
            "true_label"
        ]
        .astype(str)
        + " → "
        + task_error_data[
            "predicted_label"
        ]
        .astype(str)
    )


    transition_order = (
        task_error_data[
            "error_transition"
        ]
        .drop_duplicates()
        .tolist()
    )


    figure, axis = plt.subplots(
        figsize=(12, 6)
    )


    x_positions = np.arange(
        len(transition_order)
    )


    for adapter_name in ADAPTER_MODELS:
        adapter_errors = (
            task_error_data[
                task_error_data[
                    "adapter_name"
                ]
                == adapter_name
            ]
            .set_index(
                "error_transition"
            )
            .reindex(
                transition_order
            )
        )


        axis.plot(
            x_positions,
            adapter_errors[
                "error_reduction"
            ],
            marker="o",
            linewidth=2,
            label=adapter_name,
        )


    axis.axhline(
        0,
        linewidth=1,
    )


    axis.set_title(
        f"Variazione dei tipi di errore — "
        f"{TASK_DISPLAY_NAMES[task_name]}"
    )

    axis.set_xlabel(
        "Transizione errata"
    )

    axis.set_ylabel(
        "Errori base − errori adapter"
    )


    axis.set_xticks(
        x_positions
    )

    axis.set_xticklabels(
        transition_order,
        rotation=35,
        ha="right",
    )


    axis.legend()

    apply_y_grid(
        axis
    )


    save_figure(
        figure,
        ERROR_TYPE_DIR
        / f"{task_name}_error_reduction.png",
    )

In [ ]:
# CELLA 66 — Visualizza la distribuzione dell'errore assoluto
# e la tendenza a sottostimare o sovrastimare in STS-B.

stsb_profile_df = (
    stsb_distance_profile_df
    .set_index("model_name")
    .reindex(MODEL_ORDER)
)


absolute_error_columns = [
    "exact_count",
    "off_by_1_count",
    "off_by_2_count",
    "off_by_3_or_more_count",
]


figure, axis = plt.subplots(
    figsize=(11, 6)
)


bottom_values = np.zeros(
    len(MODEL_ORDER)
)


for column_name in absolute_error_columns:
    values = (
        stsb_profile_df[
            column_name
        ]
        .to_numpy(
            dtype=float
        )
    )

    axis.bar(
        MODEL_ORDER,
        values,
        bottom=bottom_values,
        label=column_name,
    )

    bottom_values += values


axis.set_title(
    "STS-B — Distribuzione della distanza assoluta"
)

axis.set_xlabel(
    "Modello"
)

axis.set_ylabel(
    "Numero di esempi"
)

axis.tick_params(
    axis="x",
    rotation=25,
)

axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    STSB_PLOTS_DIR
    / "stsb_absolute_error_distribution.png",
)


signed_columns = [
    "underestimation_count",
    "exact_prediction_count",
    "overestimation_count",
]


figure, axis = plt.subplots(
    figsize=(11, 6)
)


bottom_values = np.zeros(
    len(MODEL_ORDER)
)


for column_name in signed_columns:
    values = (
        stsb_profile_df[
            column_name
        ]
        .to_numpy(
            dtype=float
        )
    )

    axis.bar(
        MODEL_ORDER,
        values,
        bottom=bottom_values,
        label=column_name,
    )

    bottom_values += values


axis.set_title(
    "STS-B — Sottostime, previsioni esatte e sovrastime"
)

axis.set_xlabel(
    "Modello"
)

axis.set_ylabel(
    "Numero di esempi"
)

axis.tick_params(
    axis="x",
    rotation=25,
)

axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    STSB_PLOTS_DIR
    / "stsb_signed_error_distribution.png",
)

In [ ]:
# CELLA 67 — Genera una matrice 6×6 tra valore reale
# e valore previsto per ciascun modello STS-B.

STSB_LABELS = [
    0,
    1,
    2,
    3,
    4,
    5,
]


for model_name in MODEL_ORDER:
    model_data = (
        stsb_results_df[
            stsb_results_df[
                "model_name"
            ]
            == model_name
        ]
    )


    matrix = (
        pd.crosstab(
            model_data[
                "y_true"
            ],
            model_data[
                "y_pred"
            ],
        )
        .reindex(
            index=STSB_LABELS,
            columns=STSB_LABELS,
            fill_value=0,
        )
        .to_numpy(
            dtype=float
        )
    )


    figure, axis = plt.subplots(
        figsize=(6, 5)
    )

    image = axis.imshow(
        matrix,
        aspect="equal",
    )


    axis.set_title(
        f"STS-B true vs predicted — {model_name}"
    )

    axis.set_xlabel(
        "Predicted score"
    )

    axis.set_ylabel(
        "True score"
    )


    axis.set_xticks(
        np.arange(
            len(STSB_LABELS)
        )
    )

    axis.set_xticklabels(
        STSB_LABELS
    )


    axis.set_yticks(
        np.arange(
            len(STSB_LABELS)
        )
    )

    axis.set_yticklabels(
        STSB_LABELS
    )


    annotate_heatmap(
        axis,
        matrix,
        decimals=0,
    )


    figure.colorbar(
        image,
        ax=axis,
        label="Numero di esempi",
    )


    save_figure(
        figure,
        STSB_PLOTS_DIR
        / f"stsb_true_vs_predicted_{model_name}.png",
    )

In [ ]:
# CELLA 68 — Visualizza la distanza assoluta dal valore corretto
# per ciascun esempio e modello STS-B.

stsb_error_matrix_df = (
    stsb_results_df
    .pivot(
        index="example_id",
        columns="model_name",
        values="absolute_error",
    )
    .reindex(
        columns=MODEL_ORDER
    )
)


figure, axis = plt.subplots(
    figsize=(14, 6)
)


for model_name in MODEL_ORDER:
    axis.plot(
        stsb_error_matrix_df.index,
        stsb_error_matrix_df[
            model_name
        ],
        marker="o",
        linewidth=1.5,
        label=model_name,
    )


axis.set_title(
    "STS-B — Distanza assoluta dal valore corretto per esempio"
)

axis.set_xlabel(
    "Example ID"
)

axis.set_ylabel(
    "Errore assoluto"
)

axis.tick_params(
    axis="x",
    rotation=60,
)

axis.set_yticks(
    STSB_LABELS
)

axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    STSB_PLOTS_DIR
    / "stsb_absolute_error_by_example.png",
)

In [ ]:
# CELLA 69 — Confronta MAE e RMSE dei cinque modelli
# nel task STS-B.

stsb_metrics_for_plot_df = (
    fine_grained_metrics_df[
        fine_grained_metrics_df[
            "task_name"
        ]
        == STSB_TASK
    ]
    [
        [
            "model_name",
            "mae",
            "rmse",
        ]
    ]
    .copy()
)


stsb_metrics_for_plot_df = sort_models(
    stsb_metrics_for_plot_df
)


figure, axis = plt.subplots(
    figsize=(10, 5)
)


x_positions = np.arange(
    len(MODEL_ORDER)
)

bar_width = 0.35


axis.bar(
    x_positions
    - bar_width / 2,
    stsb_metrics_for_plot_df[
        "mae"
    ],
    width=bar_width,
    label="MAE",
)

axis.bar(
    x_positions
    + bar_width / 2,
    stsb_metrics_for_plot_df[
        "rmse"
    ],
    width=bar_width,
    label="RMSE",
)


axis.set_title(
    "STS-B — MAE e RMSE"
)

axis.set_xlabel(
    "Modello"
)

axis.set_ylabel(
    "Errore"
)


axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    MODEL_ORDER,
    rotation=25,
    ha="right",
)


axis.legend()

apply_y_grid(
    axis
)


save_figure(
    figure,
    STSB_PLOTS_DIR
    / "stsb_mae_rmse.png",
)

In [ ]:
# CELLA 70 — Definisce i file CSV e JSON prodotti
# dal notebook finale.

OUTPUT_PATHS = {
    "automatic_primary_scores": (
        FINAL_TABLES_DIR
        / "automatic_primary_scores.csv"
    ),

    "task_rankings_automatic": (
        FINAL_TABLES_DIR
        / "task_rankings_automatic.csv"
    ),

    "glue_primary_scores_5x7": (
        FINAL_TABLES_DIR
        / "glue_primary_scores_5x7.csv"
    ),

    "task_type_summary_5x3": (
        FINAL_TABLES_DIR
        / "task_type_summary_5x3.csv"
    ),

    "base_vs_finetuning_absolute": (
        FINAL_TABLES_DIR
        / "base_vs_finetuning_absolute.csv"
    ),

    "base_vs_finetuning_delta": (
        FINAL_TABLES_DIR
        / "base_vs_finetuning_delta.csv"
    ),

    "win_tie_loss_vs_base": (
        FINAL_TABLES_DIR
        / "win_tie_loss_vs_base.csv"
    ),

    "legend_variants_comparison": (
        FINAL_TABLES_DIR
        / "legend_variants_comparison.csv"
    ),

    "legend_ranking_summary": (
        FINAL_TABLES_DIR
        / "legend_ranking_summary.csv"
    ),

    "regulation_vs_legends_absolute": (
        FINAL_TABLES_DIR
        / "regulation_vs_legends_absolute.csv"
    ),

    "regulation_vs_legends_delta": (
        FINAL_TABLES_DIR
        / "regulation_vs_legends_delta.csv"
    ),

    "generation_metrics": (
        FINAL_TABLES_DIR
        / "generation_metrics.csv"
    ),

    "generation_metrics_normalized": (
        FINAL_TABLES_DIR
        / "generation_metrics_normalized.csv"
    ),

    "classification_ranking": (
        FINAL_TABLES_DIR
        / "classification_ranking.csv"
    ),

    "overall_ranking_automatic": (
        FINAL_TABLES_DIR
        / "overall_ranking_automatic.csv"
    ),

    "task_points_matrix": (
        FINAL_TABLES_DIR
        / "task_points_matrix.csv"
    ),

    "ranking_stability": (
        FINAL_TABLES_DIR
        / "ranking_stability.csv"
    ),

    "dominance_matrix": (
        FINAL_TABLES_DIR
        / "dominance_matrix.csv"
    ),

    "tie_matrix": (
        FINAL_TABLES_DIR
        / "tie_matrix.csv"
    ),

    "subjective_status": (
        FINAL_TABLES_DIR
        / "subjective_status.csv"
    ),

    "metadata": (
        FINAL_TABLES_DIR
        / "notebook_08_metadata.json"
    ),
}


if subjective_available:
    OUTPUT_PATHS.update({
        "subjective_by_example": (
            FINAL_TABLES_DIR
            / "subjective_by_example.csv"
        ),

        "subjective_by_model": (
            FINAL_TABLES_DIR
            / "subjective_by_model.csv"
        ),

        "subjective_agreement": (
            FINAL_TABLES_DIR
            / "subjective_agreement.csv"
        ),

        "subjective_generation_ranking": (
            FINAL_TABLES_DIR
            / "subjective_generation_ranking.csv"
        ),

        "automatic_vs_subjective_by_model": (
            FINAL_TABLES_DIR
            / "automatic_vs_subjective_by_model.csv"
        ),

        "automatic_vs_subjective_by_example": (
            FINAL_TABLES_DIR
            / "automatic_vs_subjective_by_example.csv"
        ),

        "overall_ranking_subjective": (
            FINAL_TABLES_DIR
            / "overall_ranking_subjective.csv"
        ),
    })

In [ ]:
# CELLA 71 — Salva tutte le tabelle finali
# in file CSV riutilizzabili nella relazione e nelle slide.

automatic_primary_scores_df.to_csv(
    OUTPUT_PATHS[
        "automatic_primary_scores"
    ],
    index=False,
    encoding="utf-8-sig",
)

task_rankings_automatic_df.to_csv(
    OUTPUT_PATHS[
        "task_rankings_automatic"
    ],
    index=False,
    encoding="utf-8-sig",
)

glue_primary_scores_matrix_df.to_csv(
    OUTPUT_PATHS[
        "glue_primary_scores_5x7"
    ],
    encoding="utf-8-sig",
)

task_type_summary_df.to_csv(
    OUTPUT_PATHS[
        "task_type_summary_5x3"
    ],
    index=False,
    encoding="utf-8-sig",
)

base_vs_finetuning_absolute_df.to_csv(
    OUTPUT_PATHS[
        "base_vs_finetuning_absolute"
    ],
    encoding="utf-8-sig",
)

base_vs_finetuning_delta_df.to_csv(
    OUTPUT_PATHS[
        "base_vs_finetuning_delta"
    ],
    encoding="utf-8-sig",
)

win_tie_loss_vs_base_df.to_csv(
    OUTPUT_PATHS[
        "win_tie_loss_vs_base"
    ],
    index=False,
    encoding="utf-8-sig",
)

legend_variants_comparison_df.to_csv(
    OUTPUT_PATHS[
        "legend_variants_comparison"
    ],
    encoding="utf-8-sig",
)

legend_ranking_summary_df.to_csv(
    OUTPUT_PATHS[
        "legend_ranking_summary"
    ],
    index=False,
    encoding="utf-8-sig",
)

regulation_vs_legends_absolute_df.to_csv(
    OUTPUT_PATHS[
        "regulation_vs_legends_absolute"
    ],
    encoding="utf-8-sig",
)

regulation_vs_legends_delta_df.to_csv(
    OUTPUT_PATHS[
        "regulation_vs_legends_delta"
    ],
    encoding="utf-8-sig",
)

generation_metrics_table_df.to_csv(
    OUTPUT_PATHS[
        "generation_metrics"
    ],
    index=False,
    encoding="utf-8-sig",
)

generation_normalized_df.to_csv(
    OUTPUT_PATHS[
        "generation_metrics_normalized"
    ],
    encoding="utf-8-sig",
)

classification_ranking_df.to_csv(
    OUTPUT_PATHS[
        "classification_ranking"
    ],
    index=False,
    encoding="utf-8-sig",
)

overall_ranking_automatic_df.to_csv(
    OUTPUT_PATHS[
        "overall_ranking_automatic"
    ],
    index=False,
    encoding="utf-8-sig",
)

task_points_matrix_df.to_csv(
    OUTPUT_PATHS[
        "task_points_matrix"
    ],
    encoding="utf-8-sig",
)

ranking_stability_df.to_csv(
    OUTPUT_PATHS[
        "ranking_stability"
    ],
    index=False,
    encoding="utf-8-sig",
)

dominance_matrix_df.to_csv(
    OUTPUT_PATHS[
        "dominance_matrix"
    ],
    encoding="utf-8-sig",
)

tie_matrix_df.to_csv(
    OUTPUT_PATHS[
        "tie_matrix"
    ],
    encoding="utf-8-sig",
)

subjective_status_df.to_csv(
    OUTPUT_PATHS[
        "subjective_status"
    ],
    index=False,
    encoding="utf-8-sig",
)


if subjective_available:
    subjective_by_example_df.to_csv(
        OUTPUT_PATHS[
            "subjective_by_example"
        ],
        index=False,
        encoding="utf-8-sig",
    )

    subjective_by_model_df.to_csv(
        OUTPUT_PATHS[
            "subjective_by_model"
        ],
        index=False,
        encoding="utf-8-sig",
    )

    subjective_agreement_df.to_csv(
        OUTPUT_PATHS[
            "subjective_agreement"
        ],
        index=False,
        encoding="utf-8-sig",
    )

    subjective_generation_ranking_df.to_csv(
        OUTPUT_PATHS[
            "subjective_generation_ranking"
        ],
        index=False,
        encoding="utf-8-sig",
    )

    automatic_vs_subjective_by_model_df.to_csv(
        OUTPUT_PATHS[
            "automatic_vs_subjective_by_model"
        ],
        index=False,
        encoding="utf-8-sig",
    )

    automatic_vs_subjective_by_example_df.to_csv(
        OUTPUT_PATHS[
            "automatic_vs_subjective_by_example"
        ],
        index=False,
        encoding="utf-8-sig",
    )

    overall_ranking_subjective_scenario_df.to_csv(
        OUTPUT_PATHS[
            "overall_ranking_subjective"
        ],
        index=False,
        encoding="utf-8-sig",
    )


print(
    "Tabelle salvate in:",
    FINAL_TABLES_DIR,
)

In [ ]:
# CELLA 72 — Salva criteri metodologici, metriche rappresentative,
# gestione delle parità e disponibilità della valutazione soggettiva.

def get_package_version(
    package_name: str,
) -> str | None:
    """Restituisce la versione installata di un pacchetto."""

    try:
        return version(
            package_name
        )

    except PackageNotFoundError:
        return None


metadata = {
    "notebook": (
        "08_rankings_plots_and_tables"
    ),

    "models": MODEL_ORDER,

    "glue_tasks": GLUE_TASKS,

    "generation_task": (
        GENERATION_TASK
    ),

    "representative_metrics": {
        "sst2_style": "accuracy",
        "paraphrase_style": (
            "mean_accuracy_f1_equivalent"
        ),
        "qqp_style": (
            "mean_accuracy_f1_duplicate"
        ),
        "mnli_style": "accuracy",
        "rte_style": "accuracy",
        "bias_detection": "macro_f1",
        "stsb_style": (
            "mean_pearson_spearman"
        ),
        "grounded_answer_generation": (
            "mean_bertscore_f1"
        ),
    },

    "ranking": {
        "points": [
            5,
            4,
            3,
            2,
            1,
        ],

        "tie_management": (
            "Average points of occupied positions"
        ),

        "tolerance": (
            RANKING_TOLERANCE
        ),
    },

    "task_type_summary": {
        "classification": (
            "Mean task-ranking points over six "
            "classification tasks"
        ),

        "numeric": (
            "STS-B ranking points"
        ),

        "generative": (
            "Automatic generation ranking points"
        ),
    },

    "subjective_evaluation": {
        "available": (
            subjective_available
        ),

        "file": (
            str(subjective_file_path)
            if subjective_file_path
            else None
        ),

        "formula": (
            "(valutazione_A + valutazione_B) / 2"
        ),

        "scenario_logic": (
            "Subjective generation ranking replaces "
            "automatic generation ranking."
        ),
    },

    "package_versions": {
        "pandas": get_package_version(
            "pandas"
        ),

        "numpy": get_package_version(
            "numpy"
        ),

        "matplotlib": get_package_version(
            "matplotlib"
        ),

        "scipy": get_package_version(
            "scipy"
        ),

        "scikit-learn": get_package_version(
            "scikit-learn"
        ),

        "openpyxl": get_package_version(
            "openpyxl"
        ),
    },
}


with OUTPUT_PATHS[
    "metadata"
].open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )


print(
    "Metadati salvati:",
    OUTPUT_PATHS[
        "metadata"
    ],
)

In [ ]:
# CELLA 73 — Mostra nel notebook le tabelle principali:
# GLUE 5×7, tipologie 5×3 e ranking aggregati.

print("\n" + "=" * 100)
print("CONFRONTO GLUE-STYLE 5×7")
print("=" * 100)

display(
    glue_primary_scores_matrix_df
    .round(4)
)


print("\n" + "=" * 100)
print("CONFRONTO PER TIPOLOGIA 5×3")
print("=" * 100)

display(
    task_type_summary_df
    .round(3)
)


print("\n" + "=" * 100)
print("RANKING CLASSIFICATORIO")
print("=" * 100)

display(
    classification_ranking_df[
        [
            "overall_position",
            "model_name",
            "total_points",
            "average_points",
            "average_rank",
            "first_places",
            "top_2_finishes",
            "last_places",
        ]
    ]
    .round(4)
)


print("\n" + "=" * 100)
print("RANKING COMPLESSIVO AUTOMATICO")
print("=" * 100)

display(
    overall_ranking_automatic_df[
        [
            "overall_position",
            "model_name",
            "total_points",
            "average_points",
            "average_rank",
            "first_places",
            "top_2_finishes",
            "last_places",
        ]
    ]
    .round(4)
)


if subjective_available:
    print("\n" + "=" * 100)
    print("RANKING COMPLESSIVO CON GENERAZIONE SOGGETTIVA")
    print("=" * 100)

    display(
        overall_ranking_subjective_scenario_df[
            [
                "overall_position",
                "model_name",
                "total_points",
                "average_points",
                "average_rank",
                "first_places",
                "top_2_finishes",
                "last_places",
            ]
        ]
        .round(4)
    )

In [ ]:
# CELLA 74 — Verifica forma delle tabelle, completezza dei ranking,
# conservazione dei 15 punti per task e creazione degli output.

if (
    glue_primary_scores_matrix_df.shape
    != (
        len(MODEL_ORDER),
        len(GLUE_TASKS),
    )
):
    raise RuntimeError(
        "La matrice GLUE non ha dimensione 5×7."
    )


if (
    task_type_summary_df.shape[0]
    != len(MODEL_ORDER)
):
    raise RuntimeError(
        "La tabella per tipologia non contiene cinque modelli."
    )


if (
    len(task_rankings_automatic_df)
    != (
        len(MODEL_ORDER)
        * len(ALL_TASKS)
    )
):
    raise RuntimeError(
        "Il ranking task-level non contiene tutte "
        "le combinazioni modello-task."
    )


points_by_task = (
    task_rankings_automatic_df
    .groupby(
        "task_name"
    )[
        "points"
    ]
    .sum()
)


invalid_task_point_mask = ~np.isclose(
    points_by_task.to_numpy(
        dtype=float
    ),
    15.0,
    atol=RANKING_TOLERANCE,
    rtol=0.0,
)


invalid_task_point_totals = (
    points_by_task.loc[
        invalid_task_point_mask
    ]
)


if not invalid_task_point_totals.empty:
    raise RuntimeError(
        "La gestione delle parità non conserva "
        "15 punti complessivi per task:\n"
        + invalid_task_point_totals.to_string()
    )


if (
    glue_primary_scores_matrix_df
    .isna()
    .any()
    .any()
):
    raise RuntimeError(
        "La matrice GLUE contiene valori mancanti."
    )


if (
    task_type_summary_df[
        [
            "classification",
            "numeric",
            "generative",
        ]
    ]
    .isna()
    .any()
    .any()
):
    raise RuntimeError(
        "La tabella 5×3 contiene valori mancanti."
    )


missing_output_files = [
    str(path)
    for path in OUTPUT_PATHS.values()
    if not path.exists()
]


if missing_output_files:
    raise RuntimeError(
        "File di output mancanti:\n- "
        + "\n- ".join(
            missing_output_files
        )
    )


generated_graphs = list(
    VISUALIZATIONS_DIR
    .rglob("*.png")
)


if not generated_graphs:
    raise RuntimeError(
        "Nessun grafico è stato generato."
    )


print("\n" + "=" * 100)
print("NOTEBOOK 08 COMPLETATO CORRETTAMENTE")
print("=" * 100)

print(
    "\nMatrice GLUE:",
    glue_primary_scores_matrix_df.shape,
)

print(
    "Tabella per tipologia:",
    task_type_summary_df.shape,
)

print(
    "Ranking task-level:",
    len(task_rankings_automatic_df),
    "righe",
)

print(
    "Valutazione soggettiva disponibile:",
    subjective_available,
)

print(
    "Grafici generati:",
    len(generated_graphs),
)

print(
    "Tabelle finali:",
    FINAL_TABLES_DIR,
)

print(
    "Visualizzazioni:",
    VISUALIZATIONS_DIR,
)

1. GLUE-style 5×7
2. Tipologie 5×3
3. Base vs fine-tuning
4. Varianti Legends
5. Regulation vs Legends
6. BLEU, ROUGE e BERTScore
7. Valutazione soggettiva
8. Ranking automatici
9. Approfondimenti diagnostici